# CS3807 Deep Learning Laboratory: Attention Mechanisms in a Pretrained ResNet50
## Experiments 8 and 9: Effect of Attention Type and Attention Position (complete lab)

This single notebook covers the whole lab sheet (`Attention_report.docx`) and replaces Parts 1 to 6.

**What it does**
1. Loads TensorFlow Flowers (5 classes), resizes to 224 x 224 x 3, applies ImageNet preprocessing, and makes a stratified 70/15/15 split (seed 42, same split as before).
2. Implements channel, spatial, sequential channel-spatial (both orders), lightweight self-attention, and the extended mechanisms SE, ECA, CBAM, and Coordinate Attention.
3. Inserts attention into a pretrained ResNet50 at P1 to P4 (single and multi-stage) and trains every configuration with the same two-phase transfer-learning schedule.
4. Records every metric the sheet asks for, for every experiment, and writes all result tables and figures to `results/`.
5. Produces the qualitative analysis: internal attention maps, channel weights, spatial heatmaps at P2/P3/P4, self-attention responses, Grad-CAM, and error analysis.

**Experiment map (22 models, 19 unique training runs)**

Some configurations are identical (B3 = A3 = D1, and E3 = A2). Each is trained once and its result is reported in every experiment table where it appears, with the `role` column marking it as a shared run.

| Experiment | Models | Fixed factor | Varied factor |
|---|---|---|---|
| Exp 1 | A0 Baseline, A1 Channel, A2 Spatial, A3 Channel+Spatial, A4 Self, A5 Combined | P3 | attention type |
| Exp 2 | B1 P1, B2 P2, B3 P3 (=A3), B4 P4 | channel+spatial | position |
| Exp 3 | C1 P2+P3, C2 P3+P4, C3 P2+P3+P4 (refs: P3 = A3, P4 = B4) | channel+spatial | multi-position |
| Exp 4 | D1 channel to spatial (=A3), D2 spatial to channel | P3 | order |
| Exp 5 | E1 k=3, E2 k=5, E3 k=7 (=A2) | P3, spatial | kernel size |
| Additional | F1 SE, F2 ECA, F3 CBAM, F4 Coordinate | P3 | established mechanisms |

**Runtime.** Each model is trained for at most 30 epochs (10 with the backbone frozen, then up to 20 with conv5_x unfrozen). Full training of 19 runs takes several hours on one GPU. Training is resumable: finished runs are detected on disk and skipped unless `CONFIG['retrain']` is set.

**Compatibility.** Written for TensorFlow 2.10 (Part 1 environment). Models are saved as weights only and rebuilt from code, which avoids custom-layer serialization problems.

## Section 0: Setup, configuration, and reproducibility

In [ ]:
# Imports, global configuration, folders, and seeds
import os, math, time, pickle, random, warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image
from scipy.stats import binomtest
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, confusion_matrix, precision_recall_fscore_support
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import ResNet50
from tensorflow.keras.applications.resnet50 import preprocess_input
try:
    from IPython.display import display
except ImportError:
    display = print

SEED = 42
random.seed(SEED); np.random.seed(SEED); tf.random.set_seed(SEED)
os.environ['PYTHONHASHSEED'] = str(SEED)
for gpu in tf.config.list_physical_devices('GPU'):
    tf.config.experimental.set_memory_growth(gpu, True)

# One configuration dictionary, shared by every model (controlled comparison)
CONFIG = dict(
    image_size=224,
    split_test_size=0.30,            # 70 / 15 / 15
    batch_size=16,
    epochs_phase1=10,                # backbone frozen: attention + head learn
    epochs_phase2=20,                # conv5_x unfrozen: fine-tuning (total <= 30)
    lr_phase1=1e-3,
    lr_phase2=1e-4,
    dropout=0.5,
    reduction_ratio=16,
    spatial_kernel=7,
    self_dim_scale=8,
    early_stop_patience=5,
    lr_patience=3,
    lr_factor=0.5,
    min_lr=1e-6,
    latency_samples=100,             # single-image timing repetitions
    bootstrap_resamples=2000,
    retrain=False,                   # True forces retraining of every run
)
IMG = CONFIG['image_size']
for d in ['results/tables', 'results/figures', 'results/models', 'results/histories', 'results/predictions']:
    os.makedirs(d, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 150, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
sns.set_theme(style='whitegrid')

def savefig(fig, name):
    path = f'results/figures/{name}'
    fig.savefig(path, dpi=150, bbox_inches='tight')
    plt.show(); plt.close(fig)
    return path

def save_table(df, name):
    df.to_csv(f'results/tables/{name}.csv', index=False)
    return df

def show(df, title):
    print(f'\n=== {title} ===')
    display(df)

print('TensorFlow', tf.__version__, '| GPUs:', len(tf.config.list_physical_devices('GPU')))
print('Configuration:', CONFIG)

## Section 1: Dataset loading and exploration

Class names are read from the dataset metadata, so label indices always match the names. The earlier notebooks hard-coded the order, which can disagree with the label indices.

In [ ]:
import tensorflow_datasets as tfds
builder = tfds.builder('tf_flowers')
builder.download_and_prepare()
CLASS_NAMES = list(builder.info.features['label'].names)
NUM_CLASSES = len(CLASS_NAMES)

raw_images, raw_labels = [], []
for ex in tfds.as_numpy(builder.as_dataset(split='train', shuffle_files=False)):
    raw_images.append(ex['image'])
    raw_labels.append(int(ex['label']))
raw_labels = np.array(raw_labels)
print(f'Loaded {len(raw_images)} images | classes (label index order): {CLASS_NAMES}')

counts = pd.Series(raw_labels).map(lambda i: CLASS_NAMES[i]).value_counts().reindex(CLASS_NAMES)
show(pd.DataFrame({'class': CLASS_NAMES, 'count': counts.values,
                   'share %': (100 * counts.values / counts.sum()).round(1)}), 'Class distribution (full dataset)')

In [ ]:
# Required visualization 1: sample images from all five classes (3 per class, raw resolution)
rng0 = np.random.default_rng(SEED)
fig, axes = plt.subplots(NUM_CLASSES, 3, figsize=(9, 2.8 * NUM_CLASSES))
for c in range(NUM_CLASSES):
    pool = np.where(raw_labels == c)[0]
    for j, i in enumerate(rng0.choice(pool, 3, replace=False)):
        axes[c, j].imshow(raw_images[i]); axes[c, j].axis('off')
        if j == 0:
            axes[c, j].set_title(CLASS_NAMES[c], loc='left', fontweight='bold')
fig.suptitle('Sample input images from all five classes (raw resolution)', y=1.0)
savefig(fig, 'F01_sample_images_all_classes.png')

## Section 2: Preprocessing, dataset split, and augmentation

Every image is resized to 224 x 224 x 3. ImageNet preprocessing (`preprocess_input`) is applied inside the input pipeline, so the stored arrays stay as compact uint8. Augmentation is defined once and used for every model.

In [ ]:
# Resize to 224 x 224 x 3 and store as uint8 (ImageNet preprocessing is applied in the tf.data pipeline)
def resize_uint8(a):
    im = Image.fromarray(np.asarray(a, dtype=np.uint8)).convert('RGB')
    return np.asarray(im.resize((IMG, IMG), Image.Resampling.LANCZOS), dtype=np.uint8)

images = np.stack([resize_uint8(a) for a in raw_images])            # (N, 224, 224, 3) uint8
onehot = keras.utils.to_categorical(raw_labels, NUM_CLASSES).astype(np.float32)
print('Resized dataset:', images.shape, images.dtype)

# Stratified split 70 / 15 / 15, seed 42 (identical to the earlier notebooks)
all_idx = np.arange(len(raw_labels))
idx_train, idx_tmp = train_test_split(all_idx, test_size=CONFIG['split_test_size'],
                                      random_state=SEED, stratify=raw_labels)
idx_val, idx_test = train_test_split(idx_tmp, test_size=0.5, random_state=SEED,
                                     stratify=raw_labels[idx_tmp])
np.savez('results/split_indices.npz', train=idx_train, val=idx_val, test=idx_test)

X_train, Y_train = images[idx_train], onehot[idx_train]
X_val, Y_val = images[idx_val], onehot[idx_val]
X_test, Y_test = images[idx_test], onehot[idx_test]
y_train, y_val, y_test = raw_labels[idx_train], raw_labels[idx_val], raw_labels[idx_test]

rows = []
for split, y in [('Training', y_train), ('Validation', y_val), ('Testing', y_test)]:
    row = {'split': split, 'samples': len(y), 'share %': round(100 * len(y) / len(raw_labels), 1)}
    row.update({CLASS_NAMES[c]: int((y == c).sum()) for c in range(NUM_CLASSES)})
    rows.append(row)
split_df = save_table(pd.DataFrame(rows), 'dataset_split_summary')
show(split_df, 'Dataset split (stratified 70 / 15 / 15)')

fig, ax = plt.subplots(figsize=(9, 4))
x = np.arange(NUM_CLASSES); w = 0.27
for k, (name, y) in enumerate([('Training', y_train), ('Validation', y_val), ('Testing', y_test)]):
    vals = [int((y == c).sum()) for c in range(NUM_CLASSES)]
    bars = ax.bar(x + (k - 1) * w, vals, w, label=name)
    ax.bar_label(bars, fontsize=7)
ax.set_xticks(x); ax.set_xticklabels(CLASS_NAMES); ax.set_ylabel('samples'); ax.legend()
ax.set_title('Class counts in each split (identical for all experiments)')
savefig(fig, 'F02_split_class_distribution.png')

In [ ]:
# Shared augmentation (applied only to training batches, identical for every model)
AUGMENT = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(15 / 360, fill_mode='nearest'),       # +/- 15 degrees
    layers.RandomZoom(0.2, fill_mode='nearest'),                # +/- 20 % zoom
    layers.RandomTranslation(0.1, 0.1, fill_mode='nearest'),    # +/- 10 % translation
    layers.RandomContrast(0.1),                                 # +/- 10 % contrast
], name='shared_augmentation')

def _prep(x, y, train):
    x = tf.cast(x, tf.float32)
    if train:
        x = AUGMENT(x, training=True)
    return preprocess_input(x), y

def make_train_ds(X, Y):
    ds = tf.data.Dataset.from_tensor_slices((X, Y)).shuffle(len(X), seed=SEED).batch(CONFIG['batch_size'])
    return ds.map(lambda x, y: _prep(x, y, True), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

def make_val_ds(X, Y):
    ds = tf.data.Dataset.from_tensor_slices((X, Y)).batch(CONFIG['batch_size'])
    return ds.map(lambda x, y: _prep(x, y, False), num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

def make_pred_ds(X):
    ds = tf.data.Dataset.from_tensor_slices(X).batch(CONFIG['batch_size'])
    return ds.map(lambda x: preprocess_input(tf.cast(x, tf.float32)),
                  num_parallel_calls=tf.data.AUTOTUNE).prefetch(tf.data.AUTOTUNE)

# Augmentation preview: one training image and five random augmentations
preview = AUGMENT(tf.cast(np.repeat(X_train[:1], 5, axis=0), tf.float32), training=True).numpy()
preview = np.clip(preview, 0, 255).astype(np.uint8)
fig, axes = plt.subplots(1, 6, figsize=(14, 2.8))
axes[0].imshow(X_train[0]); axes[0].set_title('original')
for k in range(5):
    axes[k + 1].imshow(preview[k]); axes[k + 1].set_title(f'augmented {k + 1}')
for ax in axes: ax.axis('off')
savefig(fig, 'F02b_augmentation_preview.png')

## Section 3: ResNet50 architecture and feature hierarchy

Tables 1 and 2 of the sheet are read directly from the loaded ImageNet ResNet50, so the reported dimensions are verified rather than typed in.

In [ ]:
base = ResNet50(weights='imagenet', include_top=False, input_shape=(IMG, IMG, 3))
with open('results/resnet50_architecture.txt', 'w') as f:
    base.summary(print_fn=lambda s: f.write(s + '\n'))
print(f'ResNet50 backbone (no top): {len(base.layers)} layers, {base.count_params():,} parameters')
print('Full architecture summary saved to results/resnet50_architecture.txt')

STAGES = [  # stage label, layer whose output is the stage output, candidate position, feature type
    ('Stem + MaxPool', 'pool1_pool', '-', 'low-level'),
    ('conv2_x', 'conv2_block3_out', 'P1', 'low-level'),
    ('conv3_x', 'conv3_block4_out', 'P2', 'intermediate'),
    ('conv4_x', 'conv4_block6_out', 'P3', 'high-level'),
    ('conv5_x', 'conv5_block3_out', 'P4', 'high-level'),
]
stage_rows = []
for label, ln, pos, ftype in STAGES:
    shp = base.get_layer(ln).output.shape
    stage_rows.append({'stage': label, 'layer': ln, 'position': pos,
                       'spatial H x W': f'{int(shp[1])} x {int(shp[2])}',
                       'channels C': int(shp[3]), 'feature type': ftype})
stage_df = save_table(pd.DataFrame(stage_rows), 'table1_resnet50_stages')
show(stage_df, 'Table 1: ResNet50 stage-wise feature dimensions')
show(save_table(stage_df[stage_df['position'] != '-'].reset_index(drop=True), 'table2_candidate_positions'),
     'Table 2: candidate attention positions')

fig, ax1 = plt.subplots(figsize=(8, 4))
labels_s = [r['stage'] for r in stage_rows]
ax1.bar(labels_s, [r['channels C'] for r in stage_rows], color='steelblue', alpha=0.75)
ax1.set_ylabel('channels C'); ax1.set_title('ResNet50 feature hierarchy: channels grow, resolution shrinks')
ax2 = ax1.twinx()
ax2.plot(labels_s, [int(r['spatial H x W'].split('x')[0]) for r in stage_rows], 'o-', color='firebrick')
ax2.set_ylabel('spatial size H (px)')
savefig(fig, 'F03_resnet50_feature_hierarchy.png')

## Section 4: Attention mechanisms (implementations)

Every module keeps the feature shape unchanged. Each one exposes its internal map (`channel_map`, `spatial_map`, or `attention_matrix`) so the internal attention maps can be visualized later.

| Mechanism | Formula | Position in code |
|---|---|---|
| Channel attention | M_c(F) = sigmoid(W2 ReLU(W1 GAP(F))), with reduction ratio r = 16 | `ChannelAttention` |
| Spatial attention | M_s(F) = sigmoid(Conv_k([mean_c F ; max_c F])) | `SpatialAttention(kernel_size=k)` |
| Channel then spatial | F_cs = M_s(F_c) * F_c, with F_c = M_c(F) * F | `ChannelSpatialAttention` |
| Spatial then channel | F_sc = M_c(F_s) * F_s, with F_s = M_s(F) * F | `SpatialChannelAttention` |
| Lightweight self-attention | A = softmax(QK^T / sqrt(d)), Y = AV, F_SA = F + gamma * reshape(Y) | `SelfAttention` |
| Combined | channel, then spatial, then self-attention | `CombinedAttention` |
| SE | squeeze (GAP), excitation (FC, ReLU, FC, sigmoid), scale | `SEAttention` |
| ECA | GAP, then 1D convolution across channels with adaptive kernel, sigmoid | `ECAAttention` |
| CBAM | channel submodule (shared MLP over avg and max), then 7x7 spatial conv | `CBAM` |
| Coordinate attention | pool along H and W, shared 1x1 conv, split, two directional gates | `CoordinateAttention` |

Note: SE and channel attention are the same block (Hu et al. define channel attention as SE). The SE row in the mechanism table therefore acts as a **sanity check**: with the same seed, the two runs should agree closely, and any difference shows the noise floor of the pipeline.

In [ ]:
class ChannelAttention(layers.Layer):
    """M_c(F) = sigmoid(W2 ReLU(W1 GAP(F))); output = F * M_c(F)."""
    def __init__(self, reduction_ratio=CONFIG['reduction_ratio'], **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio = reduction_ratio

    def build(self, input_shape):
        C = int(input_shape[-1])
        hidden = max(C // self.reduction_ratio, 1)
        self.gap = layers.GlobalAveragePooling2D(keepdims=True)
        self.fc1 = layers.Dense(hidden, activation='relu', name=f'{self.name}_W1')
        self.fc2 = layers.Dense(C, activation='sigmoid', name=f'{self.name}_W2')
        super().build(input_shape)

    def _mc(self, x):                              # (B, 1, 1, C)
        return self.fc2(self.fc1(self.gap(x)))

    def channel_map(self, x):                      # (B, C) channel weights
        return tf.squeeze(self._mc(x), axis=[1, 2])

    def call(self, x):
        return x * self._mc(x)


class SEAttention(ChannelAttention):
    """Squeeze-and-Excitation (Hu et al.): squeeze = GAP, excitation = FC-ReLU-FC-sigmoid, scale."""
    pass


class SpatialAttention(layers.Layer):
    """M_s(F) = sigmoid(Conv_k([mean_c F ; max_c F])); output = F * M_s(F)."""
    def __init__(self, kernel_size=CONFIG['spatial_kernel'], **kwargs):
        super().__init__(**kwargs)
        self.kernel_size = kernel_size

    def build(self, input_shape):
        self.conv = layers.Conv2D(1, self.kernel_size, padding='same', activation='sigmoid',
                                  name=f'{self.name}_conv{self.kernel_size}x{self.kernel_size}')
        super().build(input_shape)

    def spatial_map(self, x):                      # (B, H, W, 1)
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx = tf.reduce_max(x, axis=-1, keepdims=True)
        return self.conv(tf.concat([avg, mx], axis=-1))

    def call(self, x):
        return x * self.spatial_map(x)


class ChannelSpatialAttention(layers.Layer):
    """Sequential: channel attention first ('what'), then spatial attention ('where')."""
    def __init__(self, reduction_ratio=CONFIG['reduction_ratio'], spatial_kernel=CONFIG['spatial_kernel'], **kwargs):
        super().__init__(**kwargs)
        self.channel_attn = ChannelAttention(reduction_ratio, name=f'{self.name}_channel')
        self.spatial_attn = SpatialAttention(spatial_kernel, name=f'{self.name}_spatial')

    def call(self, x):
        return self.spatial_attn(self.channel_attn(x))


class SpatialChannelAttention(layers.Layer):
    """Reverse order (Experiment 4): spatial attention first, then channel attention."""
    def __init__(self, reduction_ratio=CONFIG['reduction_ratio'], spatial_kernel=CONFIG['spatial_kernel'], **kwargs):
        super().__init__(**kwargs)
        self.spatial_attn = SpatialAttention(spatial_kernel, name=f'{self.name}_spatial')
        self.channel_attn = ChannelAttention(reduction_ratio, name=f'{self.name}_channel')

    def call(self, x):
        return self.channel_attn(self.spatial_attn(x))


class SelfAttention(layers.Layer):
    """Lightweight non-local self-attention over the N = H*W spatial positions, with learnable residual gamma."""
    def __init__(self, dim_scale=CONFIG['self_dim_scale'], **kwargs):
        super().__init__(**kwargs)
        self.dim_scale = dim_scale

    def build(self, input_shape):
        self.H, self.W, self.C = int(input_shape[1]), int(input_shape[2]), int(input_shape[3])
        self.d = max(self.C // self.dim_scale, 1)
        self.q_proj = layers.Dense(self.d, name=f'{self.name}_q')
        self.k_proj = layers.Dense(self.d, name=f'{self.name}_k')
        self.v_proj = layers.Dense(self.d, name=f'{self.name}_v')
        self.o_proj = layers.Dense(self.C, name=f'{self.name}_o')
        self.gamma = self.add_weight(name='gamma', shape=(), initializer='zeros', trainable=True)
        super().build(input_shape)

    def _attend(self, x):
        b = tf.shape(x)[0]
        xf = tf.reshape(x, [b, self.H * self.W, self.C])          # (B, N, C)
        q, k, v = self.q_proj(xf), self.k_proj(xf), self.v_proj(xf)
        A = tf.nn.softmax(tf.matmul(q, k, transpose_b=True) / tf.sqrt(float(self.d)), axis=-1)   # (B, N, N)
        y = self.o_proj(tf.matmul(A, v))                           # (B, N, C)
        return A, y

    def attention_matrix(self, x):                                  # (B, N, N)
        return self._attend(x)[0]

    def call(self, x):
        _, y = self._attend(x)
        y = tf.reshape(y, [tf.shape(x)[0], self.H, self.W, self.C])
        return x + self.gamma * y


class CombinedAttention(layers.Layer):
    """Channel -> spatial -> self-attention (Model A5)."""
    def __init__(self, reduction_ratio=CONFIG['reduction_ratio'], spatial_kernel=CONFIG['spatial_kernel'],
                 dim_scale=CONFIG['self_dim_scale'], **kwargs):
        super().__init__(**kwargs)
        self.channel_attn = ChannelAttention(reduction_ratio, name=f'{self.name}_channel')
        self.spatial_attn = SpatialAttention(spatial_kernel, name=f'{self.name}_spatial')
        self.self_attn = SelfAttention(dim_scale, name=f'{self.name}_self')

    def call(self, x):
        return self.self_attn(self.spatial_attn(self.channel_attn(x)))


class ECAAttention(layers.Layer):
    """ECA: GAP, 1D convolution across channels with adaptive kernel k = |log2(C)/gamma + b/gamma|_odd."""
    def __init__(self, gamma=2, b=1, **kwargs):
        super().__init__(**kwargs)
        self.gamma_, self.b = gamma, b

    def build(self, input_shape):
        C = int(input_shape[-1])
        k = int(abs(math.log2(C) / self.gamma_ + self.b / self.gamma_))
        self.k_size = k if k % 2 == 1 else k + 1
        self.gap = layers.GlobalAveragePooling2D()
        self.conv1d = layers.Conv1D(1, self.k_size, padding='same', use_bias=False, name=f'{self.name}_conv1d')
        super().build(input_shape)

    def channel_map(self, x):                                       # (B, C)
        z = self.gap(x)[:, :, None]                                 # (B, C, 1)
        return tf.nn.sigmoid(tf.squeeze(self.conv1d(z), axis=-1))

    def call(self, x):
        return x * self.channel_map(x)[:, None, None, :]


class CBAM(layers.Layer):
    """CBAM: channel submodule (shared MLP over avg and max descriptors), then spatial submodule (7x7 conv)."""
    def __init__(self, reduction_ratio=CONFIG['reduction_ratio'], spatial_kernel=CONFIG['spatial_kernel'], **kwargs):
        super().__init__(**kwargs)
        self.reduction_ratio, self.spatial_kernel = reduction_ratio, spatial_kernel

    def build(self, input_shape):
        C = int(input_shape[-1])
        hidden = max(C // self.reduction_ratio, 1)
        self.mlp1 = layers.Dense(hidden, activation='relu', name=f'{self.name}_mlp1')
        self.mlp2 = layers.Dense(C, name=f'{self.name}_mlp2')
        self.spatial_conv = layers.Conv2D(1, self.spatial_kernel, padding='same', activation='sigmoid',
                                          name=f'{self.name}_spatial')
        super().build(input_shape)

    def channel_map(self, x):                                       # (B, C)
        avg = tf.reduce_mean(x, axis=[1, 2], keepdims=True)
        mx = tf.reduce_max(x, axis=[1, 2], keepdims=True)
        m = tf.nn.sigmoid(self.mlp2(self.mlp1(avg)) + self.mlp2(self.mlp1(mx)))
        return tf.squeeze(m, axis=[1, 2])

    def spatial_map(self, x):                                       # (B, H, W, 1)
        avg = tf.reduce_mean(x, axis=-1, keepdims=True)
        mx = tf.reduce_max(x, axis=-1, keepdims=True)
        return self.spatial_conv(tf.concat([avg, mx], axis=-1))

    def call(self, x):
        xc = x * self.channel_map(x)[:, None, None, :]
        return xc * self.spatial_map(xc)


class CoordinateAttention(layers.Layer):
    """Coordinate attention (Hou et al.): direction-aware H-pool and W-pool gates."""
    def __init__(self, reduction=32, **kwargs):
        super().__init__(**kwargs)
        self.reduction = reduction

    def build(self, input_shape):
        C = int(input_shape[-1])
        mid = max(C // self.reduction, 8)
        self.reduce = layers.Conv2D(mid, 1, use_bias=False, name=f'{self.name}_reduce')
        self.bn = layers.BatchNormalization(name=f'{self.name}_bn')
        self.conv_h = layers.Conv2D(C, 1, activation='sigmoid', name=f'{self.name}_h')
        self.conv_w = layers.Conv2D(C, 1, activation='sigmoid', name=f'{self.name}_w')
        super().build(input_shape)

    def _gates(self, x):
        H = int(x.shape[1])
        zh = tf.reduce_mean(x, axis=2, keepdims=True)                              # (B, H, 1, C)
        zw = tf.transpose(tf.reduce_mean(x, axis=1, keepdims=True), [0, 2, 1, 3])  # (B, W, 1, C)
        z = tf.concat([zh, zw], axis=1)                                            # (B, H+W, 1, C)
        z = self.bn(self.reduce(z))
        z = z * tf.nn.relu6(z + 3.0) / 6.0                                         # h-swish
        gh = self.conv_h(z[:, :H])                                                 # (B, H, 1, C)
        gw = tf.transpose(self.conv_w(z[:, H:]), [0, 2, 1, 3])                     # (B, 1, W, C)
        return gh, gw

    def call(self, x):
        gh, gw = self._gates(x)
        return x * gh * gw


def make_attention(kind, name, kernel=CONFIG['spatial_kernel']):
    r, s = CONFIG['reduction_ratio'], CONFIG['self_dim_scale']
    if kind == 'channel':         return ChannelAttention(r, name=name)
    if kind == 'spatial':         return SpatialAttention(kernel, name=name)
    if kind == 'channel_spatial': return ChannelSpatialAttention(r, kernel, name=name)
    if kind == 'spatial_channel': return SpatialChannelAttention(r, kernel, name=name)
    if kind == 'self':            return SelfAttention(s, name=name)
    if kind == 'combined':        return CombinedAttention(r, kernel, s, name=name)
    if kind == 'se':              return SEAttention(r, name=name)
    if kind == 'eca':             return ECAAttention(name=name)
    if kind == 'cbam':            return CBAM(r, kernel, name=name)
    if kind == 'coordinate':      return CoordinateAttention(32, name=name)
    raise ValueError(f'unknown attention kind: {kind}')

print('All attention modules defined.')

### Unit tests and module cost at P2, P3, and P4

Each module is tested for shape preservation and parameter count at all three stage outputs (P2 = 28x28x512, P3 = 14x14x1024, P4 = 7x7x2048). The residual self-attention must be the identity at initialization, because gamma starts at 0.

In [ ]:
TEST_SHAPES = {'P2': (28, 28, 512), 'P3': (14, 14, 1024), 'P4': (7, 7, 2048)}
ATTN_KINDS = ['channel', 'spatial', 'channel_spatial', 'spatial_channel', 'self', 'combined', 'se', 'eca', 'cbam', 'coordinate']
rows = []
for pos, (h, w, c) in TEST_SHAPES.items():
    x = tf.random.normal((2, h, w, c))
    for kind in ATTN_KINDS:
        m = make_attention(kind, f'test_{pos}_{kind}')
        y = m(x, training=False)
        rows.append({'position': pos, 'attention': kind, 'input H x W x C': f'{h} x {w} x {c}',
                     'output H x W x C': 'x'.join(str(int(d)) for d in y.shape[1:]),
                     'shape preserved': tuple(y.shape) == tuple(x.shape),
                     'parameters': int(m.count_params())})
cost_df = save_table(pd.DataFrame(rows), 'attention_module_cost_by_position')
show(cost_df, 'Attention module shapes and parameter counts at P2, P3, P4')
assert cost_df['shape preserved'].all(), 'an attention module changed the feature shape'

sa = make_attention('self', 'check_self')
x_chk = tf.random.normal((1, 14, 14, 1024))
print('Self-attention identity at init (max |F_SA - F|):', float(np.abs((sa(x_chk) - x_chk).numpy()).max()))

# Figure F04: parameter cost of every module at P3
p3 = cost_df[cost_df['position'] == 'P3'].sort_values('parameters')
fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(p3['attention'], p3['parameters'], color='coral')
ax.bar_label(bars, labels=[f'{v:,}' for v in p3['parameters']], fontsize=8, padding=2)
ax.set_xscale('log'); ax.set_xlabel('parameters (log scale)')
ax.set_title('Parameter cost of each attention module at P3 (14 x 14 x 1024)')
savefig(fig, 'F04_attention_module_params_P3.png')

## Section 5: Building ResNet50 with attention inserted at a chosen stage

The attention module is inserted with `keras.models.clone_model`. Each ResNet50 stage output that is selected is replaced by a `PositionAttention` wrapper, which applies the block's ReLU and then the attention module. All other layers are cloned unchanged, and the pretrained ImageNet weights are copied into them. The attention module and the classifier head start from the same seed in every run.

**Two-phase transfer learning (identical for all models)**
- Phase 1: the pretrained backbone is frozen. Only attention, GAP, Dropout, and the 5-way classifier train (lr = 1e-3).
- Phase 2: conv5_x is unfrozen and the network is fine-tuned with a smaller learning rate (1e-4). BatchNorm layers stay frozen.

In [ ]:
POS_LAYER = {'P1': 'conv2_block3_out', 'P2': 'conv3_block4_out', 'P3': 'conv4_block6_out', 'P4': 'conv5_block3_out'}
LAYER_POS = {v: k for k, v in POS_LAYER.items()}
HEAD_NAMES = ('gap', 'dropout', 'predictions')

class PositionAttention(layers.Layer):
    """Replaces one ResNet50 stage output: the block's ReLU, then the attention module."""
    def __init__(self, attn, **kwargs):
        super().__init__(**kwargs)
        self.act = layers.Activation('relu')
        self.attn = attn

    def call(self, x):
        return self.attn(self.act(x))


def build_model(kind='none', positions=('P3',), kernel=CONFIG['spatial_kernel'],
                pretrained=True, seed=SEED, run_name='model'):
    tf.random.set_seed(seed)
    base_m = ResNet50(weights='imagenet' if pretrained else None, include_top=False,
                      input_shape=(IMG, IMG, 3))
    targets = {POS_LAYER[p]: p for p in positions}

    def clone_fn(layer):
        if kind != 'none' and layer.name in targets:
            p = targets[layer.name]
            return PositionAttention(make_attention(kind, f'{p}_{kind}', kernel), name=layer.name)
        return layer.__class__.from_config(layer.get_config())

    body = keras.models.clone_model(base_m, clone_function=clone_fn)
    if pretrained:                                    # copy ImageNet weights into the cloned backbone
        for bl in base_m.layers:
            if bl.weights:
                body.get_layer(bl.name).set_weights(bl.get_weights())
    x = layers.GlobalAveragePooling2D(name='gap')(body.output)
    x = layers.Dropout(CONFIG['dropout'], name='dropout')(x)
    out = layers.Dense(NUM_CLASSES, activation='softmax', name='predictions')(x)
    return keras.Model(body.input, out, name=run_name)


def set_phase(model, phase):
    """Phase 1: backbone frozen. Phase 2: conv5_x conv layers unfrozen. BatchNorm always frozen."""
    for l in model.layers:
        if isinstance(l, layers.InputLayer):
            continue
        if isinstance(l, PositionAttention) or l.name in HEAD_NAMES:
            l.trainable = True
        elif isinstance(l, layers.BatchNormalization):
            l.trainable = False
        elif l.name.startswith('conv5_'):
            l.trainable = (phase == 2)
        else:
            l.trainable = False


def trainable_count(model):
    return int(sum(np.prod(tuple(v.shape)) for v in model.trainable_weights))

def attention_params(model):
    return int(sum(l.attn.count_params() for l in model.layers if isinstance(l, PositionAttention)))

def attention_io_records(model, run):
    recs = []
    for l in model.layers:
        if isinstance(l, PositionAttention):
            i = tuple(int(d) for d in l.input.shape[1:]); o = tuple(int(d) for d in l.output.shape[1:])
            recs.append({'run': run, 'position': LAYER_POS[l.name], 'attention': RUN_SPECS[run]['kind'],
                         'input H x W x C': 'x'.join(map(str, i)), 'output H x W x C': 'x'.join(map(str, o)),
                         'attention parameters': int(l.attn.count_params())})
    return recs

tf.keras.backend.clear_session()

## Section 6: Experiment registry and protocol

`RUN_SPECS` defines each unique trained network. `MODEL_TABLE` maps all 22 models of the sheet onto these runs, so an identical configuration appears in more than one experiment without being trained twice.

In [ ]:
RUN_SPECS = {
    'A0_Baseline':       dict(kind='none',            positions=()),
    'A1_Channel':        dict(kind='channel',         positions=('P3',)),
    'A2_Spatial':        dict(kind='spatial',         positions=('P3',), kernel=7),
    'A3_ChannelSpatial': dict(kind='channel_spatial', positions=('P3',)),
    'A4_Self':           dict(kind='self',            positions=('P3',)),
    'A5_Combined':       dict(kind='combined',        positions=('P3',)),
    'B1_P1':             dict(kind='channel_spatial', positions=('P1',)),
    'B2_P2':             dict(kind='channel_spatial', positions=('P2',)),
    'B4_P4':             dict(kind='channel_spatial', positions=('P4',)),
    'C1_P2P3':           dict(kind='channel_spatial', positions=('P2', 'P3')),
    'C2_P3P4':           dict(kind='channel_spatial', positions=('P3', 'P4')),
    'C3_P2P3P4':         dict(kind='channel_spatial', positions=('P2', 'P3', 'P4')),
    'D2_SpatialChannel': dict(kind='spatial_channel', positions=('P3',)),
    'E1_Kernel3':        dict(kind='spatial',         positions=('P3',), kernel=3),
    'E2_Kernel5':        dict(kind='spatial',         positions=('P3',), kernel=5),
    'F1_SE':             dict(kind='se',              positions=('P3',)),
    'F2_ECA':            dict(kind='eca',             positions=('P3',)),
    'F3_CBAM':           dict(kind='cbam',            positions=('P3',)),
    'F4_Coordinate':     dict(kind='coordinate',      positions=('P3',)),
}
MODEL_TABLE = [
    ('A0_Baseline',        'Exp 1: attention type', 'A0_Baseline',        'tested',     'Pretrained ResNet50, no attention (reference)'),
    ('A1_Channel',         'Exp 1: attention type', 'A1_Channel',         'tested',     'Channel attention @P3'),
    ('A2_Spatial',         'Exp 1: attention type', 'A2_Spatial',         'tested',     'Spatial attention @P3 (7x7)'),
    ('A3_ChannelSpatial',  'Exp 1: attention type', 'A3_ChannelSpatial',  'tested',     'Channel -> Spatial @P3'),
    ('A4_Self',            'Exp 1: attention type', 'A4_Self',            'tested',     'Lightweight self-attention @P3'),
    ('A5_Combined',        'Exp 1: attention type', 'A5_Combined',        'tested',     'Channel -> Spatial -> Self @P3'),
    ('B1_P1',              'Exp 2: position',       'B1_P1',              'tested',     'Channel -> Spatial @P1 (56 x 56)'),
    ('B2_P2',              'Exp 2: position',       'B2_P2',              'tested',     'Channel -> Spatial @P2'),
    ('B3_P3',              'Exp 2: position',       'A3_ChannelSpatial',  'shared run', 'Channel -> Spatial @P3 (same config as A3)'),
    ('B4_P4',              'Exp 2: position',       'B4_P4',              'tested',     'Channel -> Spatial @P4'),
    ('C1_P2P3',            'Exp 3: multi-position', 'C1_P2P3',            'tested',     'Channel -> Spatial @P2 and @P3'),
    ('C2_P3P4',            'Exp 3: multi-position', 'C2_P3P4',            'tested',     'Channel -> Spatial @P3 and @P4'),
    ('C3_P2P3P4',          'Exp 3: multi-position', 'C3_P2P3P4',          'tested',     'Channel -> Spatial @P2, @P3 and @P4'),
    ('D1_ChannelSpatial',  'Exp 4: order',          'A3_ChannelSpatial',  'shared run', 'Channel -> Spatial @P3 (same config as A3)'),
    ('D2_SpatialChannel',  'Exp 4: order',          'D2_SpatialChannel',  'tested',     'Spatial -> Channel @P3'),
    ('E1_Kernel3',         'Exp 5: kernel',         'E1_Kernel3',         'tested',     'Spatial attention 3x3 @P3'),
    ('E2_Kernel5',         'Exp 5: kernel',         'E2_Kernel5',         'tested',     'Spatial attention 5x5 @P3'),
    ('E3_Kernel7',         'Exp 5: kernel',         'A2_Spatial',         'shared run', 'Spatial attention 7x7 @P3 (same config as A2)'),
    ('F1_SE',              'Additional mechanisms', 'F1_SE',              'tested',     'Squeeze-and-Excitation @P3'),
    ('F2_ECA',             'Additional mechanisms', 'F2_ECA',             'tested',     'Efficient Channel Attention @P3'),
    ('F3_CBAM',            'Additional mechanisms', 'F3_CBAM',            'tested',     'CBAM (channel -> spatial) @P3'),
    ('F4_Coordinate',      'Additional mechanisms', 'F4_Coordinate',      'tested',     'Coordinate Attention @P3'),
]
MODEL_TABLE_DF = pd.DataFrame(MODEL_TABLE, columns=['model', 'experiment', 'run', 'role', 'description'])
assert all(r in RUN_SPECS for r in MODEL_TABLE_DF['run']), 'unknown run in MODEL_TABLE'
print(f'{len(MODEL_TABLE_DF)} models map onto {MODEL_TABLE_DF["run"].nunique()} unique training runs')
show(MODEL_TABLE_DF, 'Experiment plan (model -> run)')

# Protocol: everything except the factor under study is identical
PROTOCOL = pd.DataFrame([
    ('dataset / split', 'stratified 70/15/15, seed 42, same indices for all runs'),
    ('preprocessing', 'resize 224x224x3, ImageNet preprocess_input'),
    ('augmentation', 'hflip, rotation +/-15 deg, zoom +/-20 %, translation +/-10 %, contrast +/-10 % (train only)'),
    ('backbone', 'ResNet50, ImageNet weights'),
    ('classifier', 'GAP -> Dropout(0.5) -> Dense(5, softmax)'),
    ('optimizer', 'Adam'),
    ('loss', 'categorical cross-entropy'),
    ('batch size', CONFIG['batch_size']),
    ('epochs', f"phase 1: {CONFIG['epochs_phase1']} (lr {CONFIG['lr_phase1']}), phase 2: {CONFIG['epochs_phase2']} (lr {CONFIG['lr_phase2']})"),
    ('early stopping / LR plateau', f"patience {CONFIG['early_stop_patience']} / {CONFIG['lr_patience']}, factor {CONFIG['lr_factor']}"),
    ('random seed', SEED),
    ('evaluation', 'same test images, accuracy, macro P/R/F1, weighted F1, losses, params, latency'),
], columns=['item', 'setting (identical for every run)'])
show(save_table(PROTOCOL, 'protocol_identical_settings'), 'Controlled protocol')

In [ ]:
# Structural check: two representative configurations
for run_test in ['A3_ChannelSpatial', 'C3_P2P3P4']:
    tf.keras.backend.clear_session()
    spec = {'kind': 'channel_spatial', 'positions': ('P3',)} if run_test == 'A3_ChannelSpatial' \
        else {'kind': 'channel_spatial', 'positions': ('P2', 'P3', 'P4')}
    m = build_model(run_name=run_test, **spec)
    set_phase(m, 1); t1 = trainable_count(m); set_phase(m, 2); t2 = trainable_count(m)
    print(f'{run_test}: total params {m.count_params():,} | trainable phase 1 {t1:,} | phase 2 {t2:,} | '
          f'attention params {attention_params(m):,}')
    for rec in attention_io_records(m, run_test):
        print('   ', rec['position'], rec['input H x W x C'], '->', rec['output H x W x C'])
tf.keras.backend.clear_session()

## Section 7: Training (all 19 unique runs)

Every run uses the same two-phase schedule, data pipeline, and seed. Finished runs are skipped, so the loop can be interrupted and resumed.

In [ ]:
TRAIN_DS = make_train_ds(X_train, Y_train)
VAL_DS = make_val_ds(X_val, Y_val)

def train_two_phase(model):
    hist = {'accuracy': [], 'val_accuracy': [], 'loss': [], 'val_loss': [], 'phase': []}
    schedule = [(1, CONFIG['lr_phase1'], CONFIG['epochs_phase1']),
                (2, CONFIG['lr_phase2'], CONFIG['epochs_phase2'])]
    for phase, lr, epochs in schedule:
        set_phase(model, phase)
        model.compile(optimizer=keras.optimizers.Adam(learning_rate=lr),
                      loss=keras.losses.CategoricalCrossentropy(), metrics=['accuracy'])
        cbs = [keras.callbacks.EarlyStopping(monitor='val_loss', patience=CONFIG['early_stop_patience'],
                                             restore_best_weights=True),
               keras.callbacks.ReduceLROnPlateau(monitor='val_loss', factor=CONFIG['lr_factor'],
                                                 patience=CONFIG['lr_patience'], min_lr=CONFIG['min_lr'])]
        h = model.fit(TRAIN_DS, validation_data=VAL_DS, epochs=epochs, callbacks=cbs, verbose=0)
        for k in ('accuracy', 'val_accuracy', 'loss', 'val_loss'):
            hist[k] += [float(v) for v in h.history[k]]
        hist['phase'] += [phase] * len(h.history['loss'])
    return hist

def train_run(run):
    w_path = f'results/models/{run}.weights.h5'
    h_path = f'results/histories/{run}.pkl'
    if os.path.exists(w_path) and os.path.exists(h_path) and not CONFIG['retrain']:
        print(f'  {run:<20s} found on disk, skipping training')
        return
    tf.keras.backend.clear_session()
    model = build_model(run_name=run, **RUN_SPECS[run])
    t0 = time.time()
    hist = train_two_phase(model)
    hist['train_seconds'] = time.time() - t0
    model.save_weights(w_path)
    with open(h_path, 'wb') as f:
        pickle.dump(hist, f)
    print(f"  {run:<20s} epochs {len(hist['loss']):2d} | final train acc {hist['accuracy'][-1]:.4f} | "
          f"best val acc {max(hist['val_accuracy']):.4f} | {hist['train_seconds'] / 60:.1f} min")
    del model
    tf.keras.backend.clear_session()

print('Training', len(RUN_SPECS), 'runs with the identical protocol:')
for run in RUN_SPECS:
    train_run(run)
print('Training stage complete.')

HIST = {}
for run in RUN_SPECS:
    with open(f'results/histories/{run}.pkl', 'rb') as f:
        HIST[run] = pickle.load(f)
show(save_table(pd.DataFrame([{
        'run': r, 'kind': RUN_SPECS[r]['kind'], 'epochs run': len(HIST[r]['loss']),
        'phase-1 epochs': int(np.sum(np.array(HIST[r]['phase']) == 1)),
        'phase-2 epochs': int(np.sum(np.array(HIST[r]['phase']) == 2)),
        'final train acc': round(HIST[r]['accuracy'][-1], 4),
        'best val acc': round(max(HIST[r]['val_accuracy']), 4),
        'best val loss': round(min(HIST[r]['val_loss']), 4),
        'train minutes': round(HIST[r]['train_seconds'] / 60, 2)} for r in RUN_SPECS]), 'training_summary'),
     'Training summary (all runs)')

## Section 8: Evaluation on the held-out test set

Each trained model is rebuilt from code, its weights are loaded, and it is evaluated one at a time (memory-safe). For every run we record:
- test accuracy with a bootstrap 95 % CI, macro precision, macro recall, macro F1, weighted F1, test loss
- training and validation accuracy and loss, measured on clean (non-augmented) images
- total, trainable (phase 1 and phase 2), and attention-only parameter counts
- average inference time per image (single image, batch size 1)
- convergence: epochs run, best epoch, epochs to reach 90 % validation accuracy, training seconds
- attention input and output feature dimensions at each insertion point
- McNemar exact test against the baseline (discordant pair counts)

In [ ]:
def predict_probs(model, X):
    return model.predict(make_pred_ds(X), verbose=0)

def cce(Y, P):
    return float(-np.mean(np.sum(Y * np.log(np.clip(P, 1e-7, 1.0)), axis=1)))

def inference_ms(model, X, n):
    x = preprocess_input(tf.cast(X[:n], tf.float32))
    for i in range(3):
        _ = model(x[i:i + 1], training=False).numpy()
    t0 = time.perf_counter()
    for i in range(n):
        _ = model(x[i:i + 1], training=False).numpy()
    return (time.perf_counter() - t0) / n * 1000.0

def bootstrap_ci(v, n=CONFIG['bootstrap_resamples'], seed=SEED):
    rng = np.random.default_rng(seed)
    idx = rng.integers(0, len(v), size=(n, len(v)))
    means = v[idx].mean(axis=1)
    lo, hi = np.percentile(means, [2.5, 97.5])
    return float(lo), float(hi)

ATTN_IO_ROWS = []

def evaluate_run(run):
    tf.keras.backend.clear_session()
    model = build_model(run_name=run, pretrained=False, **RUN_SPECS[run])
    model.load_weights(f'results/models/{run}.weights.h5')
    hist = HIST[run]
    P_tr = predict_probs(model, X_train)
    P_va = predict_probs(model, X_val)
    P_te = predict_probs(model, X_test)
    pred_te = P_te.argmax(1)
    prec, rec, f1, _ = precision_recall_fscore_support(y_test, pred_te, average='macro', zero_division=0)
    _, _, f1w, _ = precision_recall_fscore_support(y_test, pred_te, average='weighted', zero_division=0)
    set_phase(model, 1); tr1 = trainable_count(model)
    set_phase(model, 2); tr2 = trainable_count(model)
    ATTN_IO_ROWS.extend(attention_io_records(model, run))
    ytr, yva = y_train, y_val
    row = {
        'test_accuracy': accuracy_score(y_test, pred_te),
        'macro_precision': prec, 'macro_recall': rec, 'macro_f1': f1, 'weighted_f1': f1w,
        'test_loss': cce(Y_test, P_te),
        'train_accuracy': accuracy_score(ytr, P_tr.argmax(1)), 'train_loss': cce(Y_train, P_tr),
        'val_accuracy': accuracy_score(yva, P_va.argmax(1)), 'val_loss': cce(Y_val, P_va),
        'total_params': int(model.count_params()),
        'trainable_params_phase1': tr1, 'trainable_params_phase2': tr2,
        'attention_params': attention_params(model),
        'inference_ms_per_image': inference_ms(model, X_test, CONFIG['latency_samples']),
        'epochs_run': len(hist['loss']),
        'best_epoch': int(np.argmin(hist['val_loss'])) + 1,
        'epochs_to_90pct_val': next((i + 1 for i, v in enumerate(hist['val_accuracy']) if v >= 0.9), np.nan),
        'train_seconds': round(hist['train_seconds'], 1),
        'attention_io_shapes': '; '.join(f"{r['position']}: {r['input H x W x C']}->{r['output H x W x C']}"
                                         for r in attention_io_records(model, run)) or '-',
    }
    tf.keras.backend.clear_session()
    return row, P_te

RUN_METRICS, PROBS = {}, {}
print('Evaluating every trained run on the held-out test set (one model in memory at a time):')
for run in RUN_SPECS:
    row, P = evaluate_run(run)
    RUN_METRICS[run] = row; PROBS[run] = P
    print(f"  {run:<20s} test acc {row['test_accuracy']:.4f} | macro F1 {row['macro_f1']:.4f} | "
          f"params {row['total_params']:,} | {row['inference_ms_per_image']:.2f} ms/img")
with open('results/predictions/test_probabilities.pkl', 'wb') as f:
    pickle.dump(PROBS, f)

# Statistical comparison against the baseline (McNemar exact test + bootstrap CI)
BASE = 'A0_Baseline'
base_ok = PROBS[BASE].argmax(1) == y_test
for run, row in RUN_METRICS.items():
    ok = PROBS[run].argmax(1) == y_test
    b = int(np.sum(base_ok & ~ok))          # baseline right, model wrong
    c = int(np.sum(~base_ok & ok))          # baseline wrong, model right
    row['mcnemar_baseline_only_correct'] = b
    row['mcnemar_model_only_correct'] = c
    row['mcnemar_p_vs_baseline'] = 1.0 if b + c == 0 else float(binomtest(min(b, c), b + c, 0.5).pvalue)
    row['test_acc_ci_low'], row['test_acc_ci_high'] = bootstrap_ci(ok.astype(float))

SPATIAL_KINDS = {'spatial', 'channel_spatial', 'spatial_channel', 'combined', 'cbam'}
def build_master():
    rows = []
    for _, r in MODEL_TABLE_DF.iterrows():
        run = r['run']; spec = RUN_SPECS[run]
        out = {'model': r['model'], 'experiment': r['experiment'], 'run': run, 'role': r['role'],
               'attention_type': spec['kind'],
               'position': '+'.join(spec['positions']) or '-',
               'kernel': spec.get('kernel', CONFIG['spatial_kernel']) if spec['kind'] in SPATIAL_KINDS else '-'}
        out.update(RUN_METRICS[run])
        rows.append(out)
    return pd.DataFrame(rows)

MASTER_COLS_FIRST = ['model', 'experiment', 'run', 'role', 'attention_type', 'position', 'kernel',
                     'test_accuracy', 'test_acc_ci_low', 'test_acc_ci_high', 'macro_precision', 'macro_recall',
                     'macro_f1', 'weighted_f1', 'test_loss', 'train_accuracy', 'train_loss', 'val_accuracy',
                     'val_loss', 'total_params', 'trainable_params_phase1', 'trainable_params_phase2',
                     'attention_params', 'inference_ms_per_image', 'epochs_run', 'best_epoch',
                     'epochs_to_90pct_val', 'train_seconds', 'attention_io_shapes',
                     'mcnemar_baseline_only_correct', 'mcnemar_model_only_correct', 'mcnemar_p_vs_baseline']
MASTER = build_master()[MASTER_COLS_FIRST]
save_table(MASTER.round(6), 'master_all_models')
ATTN_IO_DF = save_table(pd.DataFrame(ATTN_IO_ROWS), 'observations_attention_io_dimensions')
show(MASTER.round(4), 'Master results table: all 22 models (every metric, every experiment)')
show(ATTN_IO_DF, 'Observations: attention input and output feature dimensions')

# Protocol verification: every run really used the identical split and config
assert len(X_test) == len(idx_test) and set(idx_test) == set(np.load('results/split_indices.npz')['test'])
print('Protocol check: identical test split for all runs;', len(MASTER), 'model rows recorded.')

# Every run must have produced predictions with the right shape
for run in RUN_SPECS:
    assert len(PROBS[run]) == len(X_test)
print('All runs evaluated with matching shapes.')

# Mechanism sanity check: channel attention and SE are the same block, so their difference is the noise floor
d_se = RUN_METRICS['F1_SE']['test_accuracy'] - RUN_METRICS['A1_Channel']['test_accuracy']
print(f'Sanity check: SE minus Channel test accuracy = {d_se:+.4f} (same block, same seed; should be near 0)')

## Section 9: Result tables

Tables are written in the exact column layout the sheet asks for (Table 5 kernel, Table 6 mechanisms, Table 7 main result). Each experiment also has its own table containing **every** recorded column.

**Model selection.** The "selected attention model" for the confusion-matrix, Grad-CAM, and internal-map analyses is chosen by **validation** accuracy among the attention models. The test set is never used for selection; the best test result is shown only for reference.

In [ ]:
ATTN_RUNS = [r for r in RUN_SPECS if RUN_SPECS[r]['kind'] != 'none']
SELECTED = max(ATTN_RUNS, key=lambda r: RUN_METRICS[r]['val_accuracy'])
BEST_TEST_REFERENCE = max(RUN_SPECS, key=lambda r: RUN_METRICS[r]['test_accuracy'])
print(f'Selected attention model (by validation accuracy): {SELECTED} '
      f'(val {RUN_METRICS[SELECTED]["val_accuracy"]:.4f}, test {RUN_METRICS[SELECTED]["test_accuracy"]:.4f})')
print(f'Reference only, highest test accuracy overall: {BEST_TEST_REFERENCE} '
      f'({RUN_METRICS[BEST_TEST_REFERENCE]["test_accuracy"]:.4f})')

def mrow(model_name, role=None):
    r = MASTER.set_index('model').loc[model_name].copy()
    if role: r['role'] = role
    r['model'] = model_name
    return r

def exp_table(names, refs=(), name=None):
    rows = [mrow(n) for n in names] + [mrow(n, 'reference') for n in refs]
    df = pd.DataFrame(rows).reset_index(drop=True)
    if name: save_table(df.round(6), name)
    return df

T1 = exp_table(['A0_Baseline','A1_Channel','A2_Spatial','A3_ChannelSpatial','A4_Self','A5_Combined'], name='table1_exp1_attention_type')
T2 = exp_table(['B1_P1','B2_P2','B3_P3','B4_P4'], name='table2_exp2_position')
T3 = exp_table(['C1_P2P3','C2_P3P4','C3_P2P3P4'], refs=['B3_P3','B4_P4'], name='table3_exp3_multiposition')
T4 = exp_table(['D1_ChannelSpatial','D2_SpatialChannel'], name='table4_exp4_order')
T5 = exp_table(['E1_Kernel3','E2_Kernel5','E3_Kernel7'], name='table5_exp5_kernel')
T6 = exp_table(['F1_SE','F2_ECA','F3_CBAM','F4_Coordinate'], refs=['A0_Baseline'], name='table6_additional_mechanisms')
for title, t in [('Table 1 (Exp 1): attention type @P3', T1), ('Table 2 (Exp 2): attention position', T2),
                 ('Table 3 (Exp 3): multi-position (refs = single-stage P3, P4)', T3),
                 ('Table 4 (Exp 4): order', T4), ('Table 5 (Exp 5): spatial kernel', T5),
                 ('Table 6 (Additional): established mechanisms (ref = baseline)', T6)]:
    show(t.round(4), title)

# Sheet Table 3: model composition (which attention blocks each model contains)
COMP = pd.DataFrame([
    {'Model': 'A0: Baseline ResNet50', 'Channel': '-', 'Spatial': '-', 'Self-Attention': '-', 'Position': '-'},
    {'Model': 'A1: Channel Attention', 'Channel': 'yes', 'Spatial': '-', 'Self-Attention': '-', 'Position': 'P3'},
    {'Model': 'A2: Spatial Attention', 'Channel': '-', 'Spatial': 'yes', 'Self-Attention': '-', 'Position': 'P3'},
    {'Model': 'A3: Channel+Spatial', 'Channel': 'yes', 'Spatial': 'yes', 'Self-Attention': '-', 'Position': 'P3'},
    {'Model': 'A4: Self-Attention', 'Channel': '-', 'Spatial': '-', 'Self-Attention': 'yes', 'Position': 'P3'},
    {'Model': 'A5: Combined Attention', 'Channel': 'yes', 'Spatial': 'yes', 'Self-Attention': 'yes', 'Position': 'P3'},
])
show(save_table(COMP, 'sheet_table3_model_composition'), 'Sheet Table 3: model composition')

# Sheet Table 5: spatial kernel size
kern = pd.DataFrame([{'Spatial Kernel': f'{k} x {k}', 'Accuracy': round(RUN_METRICS[r]['test_accuracy'], 4),
                      'Macro F1': round(RUN_METRICS[r]['macro_f1'], 4),
                      'Inference Time (ms/img)': round(RUN_METRICS[r]['inference_ms_per_image'], 3),
                      'Parameters': RUN_METRICS[r]['total_params']}
                     for k, r in [(3, 'E1_Kernel3'), (5, 'E2_Kernel5'), (7, 'A2_Spatial')]])
show(save_table(kern, 'sheet_table5_kernel_size'), 'Sheet Table 5: Spatial kernel size')

# Sheet Table 6: attention mechanism comparison
MECH_ROWS = [('No Attention', 'A0_Baseline'), ('Channel attention', 'A1_Channel'), ('Channel / SE', 'F1_SE'),
             ('Spatial Attention', 'A2_Spatial'), ('Channel+Spatial', 'A3_ChannelSpatial'), ('ECA', 'F2_ECA'),
             ('CBAM', 'F3_CBAM'), ('Self-Attention', 'A4_Self'), ('Combined (C+S+Self)', 'A5_Combined'),
             ('Additional (Coordinate)', 'F4_Coordinate')]
mech = pd.DataFrame([{'Attention': lab, 'Accuracy': round(RUN_METRICS[r]['test_accuracy'], 4),
                      'Precision': round(RUN_METRICS[r]['macro_precision'], 4),
                      'Recall': round(RUN_METRICS[r]['macro_recall'], 4),
                      'Macro F1': round(RUN_METRICS[r]['macro_f1'], 4),
                      'Parameters': RUN_METRICS[r]['total_params'],
                      'Attention Parameters': RUN_METRICS[r]['attention_params'],
                      'Inference Time (ms/img)': round(RUN_METRICS[r]['inference_ms_per_image'], 3)}
                     for lab, r in MECH_ROWS])
show(save_table(mech, 'sheet_table6_mechanism_comparison'), 'Sheet Table 6: Attention mechanism comparison')

# Sheet Table 7: required main result table (with extra rows for the additional mechanisms)
MAIN_ROWS = [('Baseline', '-', 'A0_Baseline'), ('Channel Attention', 'P3', 'A1_Channel'),
             ('Spatial Attention', 'P3', 'A2_Spatial'), ('Channel+Spatial', 'P2', 'B2_P2'),
             ('Channel+Spatial', 'P3', 'A3_ChannelSpatial'), ('Channel+Spatial', 'P4', 'B4_P4'),
             ('Self-Attention', 'P3', 'A4_Self'), ('Additional Attention-1 (ECA)', 'P3', 'F2_ECA'),
             ('Additional Attention-2 (CBAM)', 'P3', 'F3_CBAM'),
             ('Extra: SE', 'P3', 'F1_SE'), ('Extra: Coordinate', 'P3', 'F4_Coordinate'),
             ('Extra: Combined (C+S+Self)', 'P3', 'A5_Combined')]
main = pd.DataFrame([{'Model': lab, 'Position': pos, 'Accuracy': round(RUN_METRICS[r]['test_accuracy'], 4),
                      'Precision': round(RUN_METRICS[r]['macro_precision'], 4),
                      'Recall': round(RUN_METRICS[r]['macro_recall'], 4),
                      'F1': round(RUN_METRICS[r]['macro_f1'], 4),
                      'Parameters': RUN_METRICS[r]['total_params'],
                      'Inference Time (ms/img)': round(RUN_METRICS[r]['inference_ms_per_image'], 3)}
                     for lab, pos, r in MAIN_ROWS])
show(save_table(main, 'main_result_table'), 'Sheet Table 7: Required main result table')

# Per-class results for the baseline and the selected model
def per_class_df(run):
    p, r, f, s = precision_recall_fscore_support(y_test, PROBS[run].argmax(1),
                                                 labels=range(NUM_CLASSES), zero_division=0)
    return pd.DataFrame({'class': CLASS_NAMES, 'precision': p.round(4), 'recall (= class accuracy)': r.round(4),
                         'f1': f.round(4), 'test support': s})
PC_BASE, PC_SEL = per_class_df(BASE), per_class_df(SELECTED)
show(save_table(PC_BASE, 'table8_per_class_baseline'), f'Per-class report: {BASE}')
show(save_table(PC_SEL, 'table9_per_class_selected'), f'Per-class report: {SELECTED}')

# Confusion-matrix counts for the two models required by the sheet
cm_base = confusion_matrix(y_test, PROBS[BASE].argmax(1), labels=range(NUM_CLASSES))
cm_sel = confusion_matrix(y_test, PROBS[SELECTED].argmax(1), labels=range(NUM_CLASSES))
save_table(pd.DataFrame(cm_base, index=CLASS_NAMES, columns=CLASS_NAMES).reset_index().rename(columns={'index': 'true \\ pred'}), 'table10_confusion_baseline')
save_table(pd.DataFrame(cm_sel, index=CLASS_NAMES, columns=CLASS_NAMES).reset_index().rename(columns={'index': 'true \\ pred'}), 'table11_confusion_selected')

# McNemar and bootstrap table for every run
stat_df = MASTER[['model', 'run', 'test_accuracy', 'test_acc_ci_low', 'test_acc_ci_high',
                  'mcnemar_baseline_only_correct', 'mcnemar_model_only_correct', 'mcnemar_p_vs_baseline']].drop_duplicates('run')
show(save_table(stat_df.round(4), 'table12_significance_vs_baseline'),
     'Significance vs baseline (bootstrap 95 % CI on test accuracy; exact McNemar)')
print('Test set size:', len(y_test), '| one-image accuracy resolution:', round(1 / len(y_test), 4))

## Section 10: Graphical performance analysis

The sheet asks for at least three graphical comparisons. Below are all of them and more: accuracy and macro-F1 per experiment, mechanism comparisons with confidence intervals, accuracy against parameters, macro-F1 against inference time, per-class results, and the confusion matrices.

In [ ]:
def grouped_bar(fname, title, names, series, ylabel='score', ylim=(0, 1.08)):
    x = np.arange(len(names)); k_n = len(series); w = 0.8 / k_n
    fig, ax = plt.subplots(figsize=(max(6, 1.35 * len(names)), 4.3))
    for k, (lab, vals) in enumerate(series.items()):
        bars = ax.bar(x + (k - (k_n - 1) / 2) * w, vals, w, label=lab)
        ax.bar_label(bars, fmt='%.3f', fontsize=7, padding=1)
    ax.set_xticks(x); ax.set_xticklabels(names, rotation=20, ha='right')
    ax.set_ylim(*ylim); ax.set_ylabel(ylabel); ax.set_title(title); ax.legend(fontsize=8)
    return savefig(fig, fname)

def series_of(df):
    return {'test accuracy': df['test_accuracy'].tolist(), 'macro F1': df['macro_f1'].tolist()}

grouped_bar('F05_exp1_attention_type.png', 'Exp 1: attention type @P3', T1['model'].str.replace('_', ' ').tolist(), series_of(T1))
grouped_bar('F06_exp2_position.png', 'Exp 2: attention position (channel+spatial)', ['P1','P2','P3','P4'], series_of(T2))
grouped_bar('F07_exp3_multiposition.png', 'Exp 3: multi-position (refs P3, P4 = single stage)',
            ['P2+P3','P3+P4','P2+P3+P4','ref P3','ref P4'], series_of(T3))
grouped_bar('F08_exp4_order.png', 'Exp 4: attention order @P3', ['channel->spatial','spatial->channel'], series_of(T4))
grouped_bar('F09_exp5_kernel.png', 'Exp 5: spatial kernel size @P3', ['3x3','5x5','7x7'], series_of(T5))
grouped_bar('F10_additional_mechanisms.png', 'Additional established mechanisms @P3 (ref = baseline)',
            ['SE','ECA','CBAM','Coordinate','ref baseline'], series_of(T6))

# Kernel-size cost: inference time and parameters
fig, axes = plt.subplots(1, 2, figsize=(10, 3.8))
kl = ['3x3', '5x5', '7x7']
axes[0].bar(kl, kern['Inference Time (ms/img)'], color='steelblue'); axes[0].set_title('Inference time vs kernel'); axes[0].set_ylabel('ms / image')
axes[1].bar(kl, kern['Parameters'], color='coral'); axes[1].set_title('Parameters vs kernel'); axes[1].set_ylabel('parameters')
savefig(fig, 'F11_kernel_cost.png')

In [ ]:
# Mechanism comparison (sheet's required bar graphs), with bootstrap 95 % CI error bars
MECH_PLOT = [('Baseline', 'A0_Baseline'), ('Channel', 'A1_Channel'), ('Spatial', 'A2_Spatial'),
             ('Channel+Spatial', 'A3_ChannelSpatial'), ('Self-Attention', 'A4_Self'),
             ('Combined', 'A5_Combined'), ('SE', 'F1_SE'), ('ECA', 'F2_ECA'), ('CBAM', 'F3_CBAM'),
             ('Coordinate', 'F4_Coordinate')]
labs = [m[0] for m in MECH_PLOT]
acc = np.array([RUN_METRICS[r]['test_accuracy'] for _, r in MECH_PLOT])
lo = np.array([RUN_METRICS[r]['test_acc_ci_low'] for _, r in MECH_PLOT])
hi = np.array([RUN_METRICS[r]['test_acc_ci_high'] for _, r in MECH_PLOT])
mf1 = np.array([RUN_METRICS[r]['macro_f1'] for _, r in MECH_PLOT])

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(labs, acc, color=['gray'] + ['steelblue'] * (len(labs) - 1))
ax.errorbar(labs, acc, yerr=[acc - lo, hi - acc], fmt='none', ecolor='black', capsize=3)
for b, v in zip(bars, acc): ax.text(b.get_x() + b.get_width() / 2, v + 0.01, f'{v:.3f}', ha='center', fontsize=8)
ax.set_ylabel('classification accuracy (test)'); ax.set_ylim(0, 1.08)
ax.set_title('Attention mechanism vs accuracy (bars: test accuracy, whiskers: 95 % bootstrap CI)')
plt.xticks(rotation=20)
savefig(fig, 'F12_mechanism_vs_accuracy.png')

fig, ax = plt.subplots(figsize=(10, 4.5))
bars = ax.bar(labs, mf1, color=['gray'] + ['seagreen'] * (len(labs) - 1))
ax.bar_label(bars, fmt='%.3f', fontsize=8, padding=2)
ax.set_ylabel('macro F1'); ax.set_ylim(0, 1.08); ax.set_title('Attention mechanism vs macro F1')
plt.xticks(rotation=20)
savefig(fig, 'F13_mechanism_vs_macro_f1.png')

# Accuracy vs parameters and macro-F1 vs inference time (one point per unique run)
RUN_DF = pd.DataFrame([{'run': r, 'kind': RUN_SPECS[r]['kind'], 'position': '+'.join(RUN_SPECS[r]['positions']) or '-',
                        **{k: RUN_METRICS[r][k] for k in ['test_accuracy', 'macro_f1', 'total_params',
                           'inference_ms_per_image', 'attention_params', 'test_acc_ci_low', 'test_acc_ci_high']}}
                       for r in RUN_SPECS])
fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(RUN_DF['total_params'] / 1e6, RUN_DF['test_accuracy'], s=50, c='steelblue')
for _, r in RUN_DF.iterrows():
    ax.annotate(r['run'], (r['total_params'] / 1e6, r['test_accuracy']), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('total model parameters (millions)'); ax.set_ylabel('test accuracy')
ax.set_title('Accuracy vs model parameters (complexity trade-off)')
savefig(fig, 'F14_accuracy_vs_params.png')

fig, ax = plt.subplots(figsize=(9, 5.5))
ax.scatter(RUN_DF['inference_ms_per_image'], RUN_DF['macro_f1'], s=50, c='darkorange')
for _, r in RUN_DF.iterrows():
    ax.annotate(r['run'], (r['inference_ms_per_image'], r['macro_f1']), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('inference time (ms per image)'); ax.set_ylabel('macro F1')
ax.set_title('Macro F1 vs inference time (cost-benefit)')
savefig(fig, 'F15_f1_vs_inference_time.png')

# Macro F1 of every run, sorted
d = RUN_DF.sort_values('macro_f1')
fig, ax = plt.subplots(figsize=(8, 6.5))
bars = ax.barh(d['run'], d['macro_f1'], color='seagreen')
ax.bar_label(bars, fmt='%.3f', fontsize=7, padding=2)
ax.set_xlabel('macro F1 (test)'); ax.set_xlim(0, 1.05); ax.set_title('Macro F1: all unique runs')
savefig(fig, 'F16_macro_f1_all_runs.png')

# Parameter composition: frozen backbone / fine-tuned conv5 / attention + head
comp = pd.DataFrame({'run': list(RUN_SPECS)})
comp['attention + head (trainable in phase 1)'] = [RUN_METRICS[r]['trainable_params_phase1'] for r in RUN_SPECS]
comp['conv5_x fine-tuned (phase 2 extra)'] = [RUN_METRICS[r]['trainable_params_phase2'] - RUN_METRICS[r]['trainable_params_phase1'] for r in RUN_SPECS]
comp['frozen backbone'] = [RUN_METRICS[r]['total_params'] - RUN_METRICS[r]['trainable_params_phase2'] for r in RUN_SPECS]
fig, ax = plt.subplots(figsize=(10, 6))
left = np.zeros(len(comp))
for col, colr in zip(['attention + head (trainable in phase 1)', 'conv5_x fine-tuned (phase 2 extra)', 'frozen backbone'],
                     ['coral', 'goldenrod', 'lightgray']):
    ax.barh(comp['run'], comp[col] / 1e6, left=left / 1e6, color=colr, label=col)
    left = left + comp[col].values
ax.set_xlabel('parameters (millions)'); ax.set_title('Parameter composition per run'); ax.legend(fontsize=8)
savefig(fig, 'F17_parameter_composition.png')

# Per-class accuracy: baseline vs selected
cls_x = np.arange(NUM_CLASSES)
fig, ax = plt.subplots(figsize=(9, 4.5))
bars1 = ax.bar(cls_x - 0.2, PC_BASE['recall (= class accuracy)'], 0.4, label=f'{BASE}', color='steelblue')
bars2 = ax.bar(cls_x + 0.2, PC_SEL['recall (= class accuracy)'], 0.4, label=f'{SELECTED}', color='seagreen')
ax.bar_label(bars1, fmt='%.2f', fontsize=7); ax.bar_label(bars2, fmt='%.2f', fontsize=7)
ax.set_xticks(cls_x); ax.set_xticklabels(CLASS_NAMES); ax.set_ylim(0, 1.1)
ax.set_ylabel('per-class accuracy (recall)'); ax.set_title('Per-class accuracy: baseline vs selected attention model'); ax.legend()
savefig(fig, 'F18_per_class_accuracy_baseline_vs_selected.png')

# Per-class F1 heatmap across every run
f1_heat = pd.DataFrame({r: precision_recall_fscore_support(y_test, PROBS[r].argmax(1),
                                                           labels=range(NUM_CLASSES), zero_division=0)[2]
                        for r in RUN_SPECS}, index=CLASS_NAMES).T
fig, ax = plt.subplots(figsize=(7, 7))
sns.heatmap(f1_heat, annot=True, fmt='.2f', cmap='YlGnBu', ax=ax, cbar_kws={'label': 'per-class F1'})
ax.set_title('Per-class F1 for every run'); ax.set_ylabel('run')
savefig(fig, 'F19_per_class_f1_heatmap_all_runs.png')
save_table(f1_heat.reset_index().rename(columns={'index': 'run'}).round(4), 'table13_per_class_f1_all_runs')

# Forest plot: test accuracy with bootstrap CI
d = RUN_DF.sort_values('test_accuracy').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(8, 6.5))
ax.errorbar(d['test_accuracy'], np.arange(len(d)), xerr=[d['test_accuracy'] - d['test_acc_ci_low'],
            d['test_acc_ci_high'] - d['test_accuracy']], fmt='o', color='steelblue', ecolor='gray', capsize=3)
ax.axvline(RUN_METRICS[BASE]['test_accuracy'], ls='--', color='red', lw=1, label='baseline')
ax.set_yticks(np.arange(len(d))); ax.set_yticklabels(d['run'], fontsize=8)
ax.set_xlabel('test accuracy with 95 % bootstrap CI'); ax.set_title('Forest plot: which differences exceed sampling noise?')
ax.legend()
savefig(fig, 'F20_forest_plot_test_accuracy.png')

In [ ]:
# Training and validation curves (sheet: train/val accuracy, train/val loss, epoch vs loss)
def curves(runs, metric, val_metric, title, fname):
    cols = min(3, len(runs)); rows = int(np.ceil(len(runs) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(4.4 * cols, 3.3 * rows), squeeze=False)
    for ax, run in zip(axes.flat, runs):
        h = HIST[run]; ep = np.arange(1, len(h[metric]) + 1)
        ax.plot(ep, h[metric], label='train'); ax.plot(ep, h[val_metric], label='validation')
        for b in np.where(np.diff(np.array(h['phase'])) > 0)[0]:
            ax.axvline(b + 1.5, ls='--', color='gray', lw=0.8)
        ax.set_title(run, fontsize=9); ax.set_xlabel('epoch'); ax.set_ylabel(metric.replace('_', ' '))
    for ax in list(axes.flat)[len(runs):]:
        ax.axis('off')
    axes.flat[0].legend(fontsize=8)
    fig.suptitle(title); fig.tight_layout()
    return savefig(fig, fname)

EXP1_RUNS = ['A0_Baseline', 'A1_Channel', 'A2_Spatial', 'A3_ChannelSpatial', 'A4_Self', 'A5_Combined']
curves(EXP1_RUNS, 'accuracy', 'val_accuracy', 'Exp 1: training and validation accuracy per epoch (dashed = phase change)', 'F21_exp1_accuracy_curves.png')
curves(EXP1_RUNS, 'loss', 'val_loss', 'Exp 1: training and validation loss per epoch', 'F22_exp1_loss_curves.png')
curves(['B1_P1', 'B2_P2', 'A3_ChannelSpatial', 'B4_P4', 'C1_P2P3', 'C2_P3P4', 'C3_P2P3P4'],
       'accuracy', 'val_accuracy', 'Training and validation accuracy: positions and multi-position runs', 'F23_position_val_curves.png')
curves(['D2_SpatialChannel', 'E1_Kernel3', 'E2_Kernel5', 'A2_Spatial', 'F1_SE', 'F2_ECA', 'F3_CBAM', 'F4_Coordinate'],
       'accuracy', 'val_accuracy', 'Training and validation accuracy: order, kernel, additional mechanisms', 'F24_order_kernel_additional_val_curves.png')

# Required: baseline vs selected attention model, accuracy and loss overlaid
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
for run, c in [(BASE, 'gray'), (SELECTED, 'seagreen')]:
    h = HIST[run]; ep = np.arange(1, len(h['accuracy']) + 1)
    axes[0].plot(ep, h['accuracy'], '--', color=c, label=f'{run} train')
    axes[0].plot(ep, h['val_accuracy'], '-', color=c, label=f'{run} validation')
    axes[1].plot(ep, h['loss'], '--', color=c, label=f'{run} train')
    axes[1].plot(ep, h['val_loss'], '-', color=c, label=f'{run} validation')
axes[0].set_title('Accuracy vs epoch: baseline vs selected'); axes[1].set_title('Loss vs epoch: baseline vs selected')
for ax in axes: ax.set_xlabel('epoch'); ax.legend(fontsize=7)
axes[0].set_ylabel('accuracy'); axes[1].set_ylabel('cross-entropy loss')
savefig(fig, 'F25_baseline_vs_selected_curves.png')

# Convergence speed: epochs to reach 90 % validation accuracy
conv = RUN_DF[['run']].copy()
conv['epochs to 90% val'] = [RUN_METRICS[r]['epochs_to_90pct_val'] for r in RUN_SPECS]
conv['best epoch'] = [RUN_METRICS[r]['best_epoch'] for r in RUN_SPECS]
show(save_table(conv, 'table14_convergence'), 'Convergence summary')

In [ ]:
# Confusion matrices: baseline and selected model (counts and row-normalized), plus every run
def cm_panel(ax, run, normalize):
    cm = confusion_matrix(y_test, PROBS[run].argmax(1), labels=range(NUM_CLASSES))
    data = cm / cm.sum(axis=1, keepdims=True) if normalize else cm
    sns.heatmap(data, annot=True, fmt='.2f' if normalize else 'd', cmap='Blues', ax=ax, cbar=False,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    ax.set_xlabel('predicted'); ax.set_ylabel('true')
    ax.set_title(f"{run} ({'row-normalized' if normalize else 'counts'})", fontsize=9)

for run, fname in [(BASE, 'F26a_confusion_matrix_baseline.png'), (SELECTED, 'F26b_confusion_matrix_selected.png')]:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
    cm_panel(axes[0], run, False); cm_panel(axes[1], run, True)
    fig.tight_layout(); savefig(fig, fname)

# Difference in confusion counts: selected minus baseline
fig, ax = plt.subplots(figsize=(6.5, 5.2))
sns.heatmap(cm_sel - cm_base, annot=True, fmt='+d', cmap='RdBu', center=0, ax=ax,
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
ax.set_xlabel('predicted'); ax.set_ylabel('true'); ax.set_title(f'Change in confusion counts: {SELECTED} minus {BASE}')
savefig(fig, 'F27_confusion_difference.png')

# Confusion matrices for all runs
n = len(RUN_SPECS); cols = 5; rows = int(np.ceil(n / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3.6 * cols, 3.3 * rows), squeeze=False)
for ax, run in zip(axes.flat, RUN_SPECS):
    cm_panel(ax, run, True)
for ax in list(axes.flat)[n:]:
    ax.axis('off')
fig.tight_layout(); savefig(fig, 'F28_confusion_matrices_all_runs.png')

## Section 11: Internal attention maps and Grad-CAM (qualitative analysis)

**Important distinction (sheet requirement).** An *internal attention map* is computed by the attention module itself during the forward pass (channel weights, spatial map, self-attention matrix). *Grad-CAM* is a post-hoc explanation: it is computed after prediction from gradients of the class score with respect to a convolutional feature map.

The visualization data is gathered in one pass: every run is loaded once, evaluated on the same set of test images, and released. The images are:
- one random test image per class plus one extra (`REP_IDX`)
- up to two images the baseline gets wrong and the selected model gets right (fixed) and the reverse (broken)

Conclusions are drawn from several images, not from one.

In [ ]:
# Visualization image set (test split only)
rng_v = np.random.default_rng(SEED)
REP_IDX = [int(rng_v.choice(np.where(y_test == c)[0])) for c in range(NUM_CLASSES)]
REP_IDX.append(int(rng_v.integers(len(y_test))))
base_pred, sel_pred = PROBS[BASE].argmax(1), PROBS[SELECTED].argmax(1)
FIXED_IDX = np.where((base_pred != y_test) & (sel_pred == y_test))[0][:2].tolist()
BROKEN_IDX = np.where((base_pred == y_test) & (sel_pred != y_test))[0][:2].tolist()
VIS_IDX = list(dict.fromkeys(REP_IDX + FIXED_IDX + BROKEN_IDX))
CASE = {i: ('rep' if i in REP_IDX else 'fixed' if i in FIXED_IDX else 'broken') for i in VIS_IDX}
print(f'Visualization images: {len(VIS_IDX)} (representative {len(REP_IDX)}, fixed by attention {len(FIXED_IDX)}, '
      f'broken by attention {len(BROKEN_IDX)})')
print('Labels:', [CLASS_NAMES[y_test[i]] for i in VIS_IDX])

def norm01(a):
    return (a - a.min()) / (a.max() - a.min() + 1e-8)

def grad_cam(model, img_uint8, layer_name='conv5_block3_out'):
    """Grad-CAM (Selvaraju et al.) on the predicted class; returns map at 224x224, class index, probabilities."""
    x = preprocess_input(tf.cast(img_uint8[None], tf.float32))
    gm = keras.Model(model.input, [model.get_layer(layer_name).output, model.output])
    with tf.GradientTape() as tape:
        conv, pred = gm(x, training=False)
        cls = tf.argmax(pred[0])
        score = tf.gather(pred[0], cls)
    grads = tape.gradient(score, conv)
    w = tf.reduce_mean(grads, axis=(1, 2), keepdims=True)
    cam = tf.nn.relu(tf.reduce_sum(w * conv, axis=-1))[0]
    cam = (cam / (tf.reduce_max(cam) + 1e-8))[..., None]
    cam = tf.image.resize(cam, (IMG, IMG)).numpy()[..., 0]
    return cam, int(cls.numpy()), pred.numpy()[0]

def collect_attention(model, x, keep_features=False):
    """Internal attention maps of every inserted module, keyed by ResNet position."""
    out = {}
    for l in model.layers:
        if not isinstance(l, PositionAttention):
            continue
        pos = LAYER_POS[l.name]
        f = tf.nn.relu(keras.Model(model.input, l.input)(x))
        a = l.attn
        d = {}
        if isinstance(a, ChannelSpatialAttention):
            d['channel'] = a.channel_attn.channel_map(f).numpy()
            fc = a.channel_attn(f)
            d['spatial'] = a.spatial_attn.spatial_map(fc).numpy()
        elif isinstance(a, SpatialChannelAttention):
            d['spatial'] = a.spatial_attn.spatial_map(f).numpy()
            fs = a.spatial_attn(f)
            d['channel'] = a.channel_attn.channel_map(fs).numpy()
        elif isinstance(a, CombinedAttention):
            d['channel'] = a.channel_attn.channel_map(f).numpy()
            fc = a.channel_attn(f)
            d['spatial'] = a.spatial_attn.spatial_map(fc).numpy()
            fs = a.spatial_attn(fc)
            d['self_matrix'] = a.self_attn.attention_matrix(fs).numpy()
        elif isinstance(a, SelfAttention):
            d['self_matrix'] = a.attention_matrix(f).numpy()
        elif isinstance(a, SpatialAttention):
            d['spatial'] = a.spatial_map(f).numpy()
        elif isinstance(a, CBAM):
            d['channel'] = a.channel_map(f).numpy()
            fc = f * a.channel_map(f)[:, None, None, :]
            d['spatial'] = a.spatial_map(fc).numpy()
        elif hasattr(a, 'channel_map'):                  # channel, SE, ECA
            d['channel'] = a.channel_map(f).numpy()
        if keep_features:
            d['before'] = f.numpy()
            d['after'] = keras.Model(model.input, l.output)(x).numpy()
        out[pos] = d
    return out

VIS = {}
print('Gathering Grad-CAM and internal attention maps (one model in memory at a time):')
for run in RUN_SPECS:
    tf.keras.backend.clear_session()
    model = build_model(run_name=run, pretrained=False, **RUN_SPECS[run])
    model.load_weights(f'results/models/{run}.weights.h5')
    cams, preds, probs = [], [], []
    for i in VIS_IDX:
        cam, cls, p = grad_cam(model, X_test[i])
        cams.append(cam); preds.append(cls); probs.append(p)
    x_vis = preprocess_input(tf.cast(X_test[VIS_IDX], tf.float32))
    att = collect_attention(model, x_vis, keep_features=(run == SELECTED))
    VIS[run] = {'cams': np.stack(cams), 'preds': np.array(preds), 'probs': np.stack(probs), 'att': att}
    print(f'  {run:<20s} done ({len(att)} attention insertion point(s))')
    del model
    tf.keras.backend.clear_session()

VIS_LABEL = [f"{CLASS_NAMES[y_test[i]]}" for i in VIS_IDX]
def panel_title(run, k):
    i = VIS_IDX[k]
    return f"pred {CLASS_NAMES[VIS[run]['preds'][k]]} ({VIS[run]['probs'][k].max():.2f})"

def overlay(ax, img, heat, title, cmap='jet'):
    ax.imshow(img); ax.imshow(heat, cmap=cmap, alpha=0.45); ax.set_title(title, fontsize=8); ax.axis('off')

### 11.1 Feature maps before and after attention (selected model)

The mean activation over channels before and after the attention module shows what the module changes. The right-most panel shows the absolute change.

In [ ]:
att_sel = VIS[SELECTED]['att']
k0 = 0                                                    # first visualization image
pos_list = list(att_sel.keys())
fig, axes = plt.subplots(len(pos_list), 4, figsize=(13, 3.2 * len(pos_list)), squeeze=False)
for r, pos in enumerate(pos_list):
    before = att_sel[pos]['before'][k0]; after = att_sel[pos]['after'][k0]
    H, W, C = before.shape
    axes[r, 0].imshow(X_test[VIS_IDX[k0]].astype(np.uint8)); axes[r, 0].set_title(f'input ({VIS_LABEL[k0]})', fontsize=8)
    axes[r, 1].imshow(norm01(before.mean(-1)), cmap='viridis')
    axes[r, 1].set_title(f'{pos} BEFORE attention\n{H}x{W}x{C} (mean over channels)', fontsize=8)
    axes[r, 2].imshow(norm01(after.mean(-1)), cmap='viridis')
    axes[r, 2].set_title(f'{pos} AFTER attention\n{H}x{W}x{C}', fontsize=8)
    axes[r, 3].imshow(norm01(np.abs(after - before).mean(-1)), cmap='magma')
    axes[r, 3].set_title('|after - before|', fontsize=8)
    for ax in axes[r]: ax.axis('off')
fig.suptitle(f'Feature map before and after attention: {SELECTED}', y=1.0)
savefig(fig, 'F29_feature_before_after_attention.png')

### 11.2 Channel-attention weights

Top-K channel weights (K = 10 and K = 20), for the pure channel model and for the selected model if it contains a channel module. The second figure shows whether the same channels dominate across different images, and the third compares how concentrated the weights are across mechanisms.

In [ ]:
def first_pos_with(run, key):
    for pos, d in VIS[run]['att'].items():
        if key in d:
            return pos, d[key]
    return None, None

CH_RUNS = [r for r in ['A1_Channel', SELECTED] if first_pos_with(r, 'channel')[0] is not None]
CH_RUNS = list(dict.fromkeys(CH_RUNS))
fig, axes = plt.subplots(len(CH_RUNS), 2, figsize=(13, 3.6 * len(CH_RUNS)), squeeze=False)
for r, run in enumerate(CH_RUNS):
    pos, W_ = first_pos_with(run, 'channel')
    w = W_[0]                                             # image 0
    for c, K in enumerate([10, 20]):
        top = np.argsort(-w)[:K]
        axes[r, c].bar(range(K), w[top], color='darkorange')
        axes[r, c].set_xticks(range(K)); axes[r, c].set_xticklabels([str(t) for t in top], rotation=60, fontsize=7)
        axes[r, c].set_xlabel('channel index (sorted by weight)'); axes[r, c].set_ylabel('channel weight M_c')
        axes[r, c].set_title(f'{run} @{pos}: top-{K} channels, image 0', fontsize=9)
fig.tight_layout(); savefig(fig, 'F30_channel_weights_topK.png')

run0 = CH_RUNS[0]
pos0, Wall = first_pos_with(run0, 'channel')
mean_w = Wall.mean(axis=0)
top20 = np.argsort(-mean_w)[:20]
fig, ax = plt.subplots(figsize=(7, 6))
sns.heatmap(Wall[:, top20].T, cmap='viridis', ax=ax, yticklabels=[str(t) for t in top20],
            xticklabels=[f'img{k}' for k in range(len(VIS_IDX))], cbar_kws={'label': 'channel weight'})
ax.set_title(f'{run0} @{pos0}: top-20 channel weights across images\n(does weighting change per image?)', fontsize=9)
ax.set_ylabel('channel index'); ax.set_xlabel('visualization image')
savefig(fig, 'F31_channel_weights_across_images.png')

fig, ax = plt.subplots(figsize=(9, 4.5))
for run in ['A1_Channel', 'F1_SE', 'F2_ECA', 'F3_CBAM', 'A3_ChannelSpatial', SELECTED]:
    pos, Wc = first_pos_with(run, 'channel')
    if Wc is None: continue
    s_ = np.sort(Wc.mean(axis=0))[::-1]
    ax.plot(np.arange(1, len(s_) + 1), s_, label=f'{run} @{pos}')
ax.set_xlabel('channel rank'); ax.set_ylabel('mean channel weight (over images)')
ax.set_title('How concentrated are the channel weights? (sorted curves)'); ax.legend(fontsize=7)
savefig(fig, 'F32_channel_weight_concentration.png')

### 11.3 Spatial-attention heatmaps and overlays at P2, P3, and P4

The spatial map at each stage, using the same images for each position. The spatial attention is computed on the channel-refined features, as in the channel-then-spatial design.

In [ ]:
SP_RUNS = [('B2_P2', 'P2'), ('A3_ChannelSpatial', 'P3'), ('B4_P4', 'P4')]
n_img = min(3, len(VIS_IDX))
fig, axes = plt.subplots(n_img, 1 + 2 * len(SP_RUNS), figsize=(2.6 * (1 + 2 * len(SP_RUNS)), 2.7 * n_img), squeeze=False)
for k in range(n_img):
    img = X_test[VIS_IDX[k]].astype(np.uint8)
    axes[k, 0].imshow(img); axes[k, 0].set_title(f'input\n{VIS_LABEL[k]}', fontsize=8); axes[k, 0].axis('off')
    for j, (run, pos) in enumerate(SP_RUNS):
        _, Ms = first_pos_with(run, 'spatial')
        heat = tf.image.resize(Ms[k][..., None], (IMG, IMG)).numpy()[..., 0]
        heat = norm01(heat)
        axes[k, 1 + 2 * j].imshow(heat, cmap='jet'); axes[k, 1 + 2 * j].axis('off')
        axes[k, 1 + 2 * j].set_title(f'Ms @{pos} heatmap', fontsize=8)
        overlay(axes[k, 2 + 2 * j], img, heat, f'Ms @{pos} overlay')
fig.tight_layout(); savefig(fig, 'F33_spatial_attention_P2_P3_P4.png')

# Spatial kernel ablation: 3x3, 5x5, 7x7 on the same image
KRUNS = [('E1_Kernel3', '3x3'), ('E2_Kernel5', '5x5'), ('A2_Spatial', '7x7')]
fig, axes = plt.subplots(2, 1 + 2 * len(KRUNS), figsize=(2.6 * (1 + 2 * len(KRUNS)), 5.6), squeeze=False)
for k in range(2):
    img = X_test[VIS_IDX[k]].astype(np.uint8)
    axes[k, 0].imshow(img); axes[k, 0].axis('off'); axes[k, 0].set_title(VIS_LABEL[k], fontsize=8)
    for j, (run, lab) in enumerate(KRUNS):
        _, Ms = first_pos_with(run, 'spatial')
        heat = norm01(tf.image.resize(Ms[k][..., None], (IMG, IMG)).numpy()[..., 0])
        axes[k, 1 + 2 * j].imshow(heat, cmap='jet'); axes[k, 1 + 2 * j].axis('off')
        axes[k, 1 + 2 * j].set_title(f'{lab} heatmap', fontsize=8)
        overlay(axes[k, 2 + 2 * j], img, heat, f'{lab} overlay')
fig.tight_layout(); savefig(fig, 'F34_spatial_kernel_size_maps.png')

# Order ablation: spatial-first (D2) vs channel-first (A3)
fig, axes = plt.subplots(2, 3, figsize=(10, 6.4), squeeze=False)
for k in range(2):
    img = X_test[VIS_IDX[k]].astype(np.uint8)
    _, Msd = first_pos_with('D2_SpatialChannel', 'spatial')
    _, Msc = first_pos_with('A3_ChannelSpatial', 'spatial')
    h_d = norm01(tf.image.resize(Msd[k][..., None], (IMG, IMG)).numpy()[..., 0])
    h_c = norm01(tf.image.resize(Msc[k][..., None], (IMG, IMG)).numpy()[..., 0])
    axes[k, 0].imshow(img); axes[k, 0].axis('off'); axes[k, 0].set_title(VIS_LABEL[k], fontsize=8)
    overlay(axes[k, 1], img, h_d, 'spatial FIRST (D2) overlay')
    overlay(axes[k, 2], img, h_c, 'channel FIRST (A3) overlay')
fig.tight_layout(); savefig(fig, 'F35_order_ablation_spatial_maps.png')

### 11.4 Self-attention response

For a chosen query location (centre of the 14 x 14 grid) the attention row is reshaped into a spatial map. The mean and the maximum over all queries summarize which positions are attended globally. The full N x N attention matrix shows the long-range interactions directly.

In [ ]:
_, Amat = first_pos_with('A4_Self', 'self_matrix')
if Amat is None:
    print('no self-attention module found')
else:
    A = Amat[0]                                              # (N, N) for image 0
    N = A.shape[0]; side = int(np.sqrt(N))
    q_center = (side // 2) * side + side // 2
    fig, axes = plt.subplots(1, 6, figsize=(19, 3.6))
    img = X_test[VIS_IDX[0]].astype(np.uint8)
    axes[0].imshow(img); axes[0].set_title(f'input ({VIS_LABEL[0]})', fontsize=8)
    axes[1].imshow(norm01(A[q_center].reshape(side, side)), cmap='jet'); axes[1].set_title(f'attention of centre query (#{q_center})', fontsize=8)
    axes[2].imshow(norm01(A.mean(0).reshape(side, side)), cmap='jet'); axes[2].set_title('mean over queries', fontsize=8)
    axes[3].imshow(norm01(A.max(0).reshape(side, side)), cmap='jet'); axes[3].set_title('max over queries', fontsize=8)
    overlay(axes[4], img, norm01(tf.image.resize(A.mean(0).reshape(side, side)[..., None], (IMG, IMG)).numpy()[..., 0]), 'mean response overlay')
    im = axes[5].imshow(A, cmap='magma'); axes[5].set_title(f'full {N}x{N} attention matrix', fontsize=8)
    fig.colorbar(im, ax=axes[5], fraction=0.046)
    for ax in axes[:5]: ax.axis('off')
    fig.suptitle('Self-attention response (A4_Self @P3, 14 x 14 grid)', y=1.02)
    savefig(fig, 'F36_self_attention_response.png')
    print(f'Self-attention: attention rows sum to {A.sum(axis=1).mean():.3f} (softmax check); '
          f'centre query max weight {A[q_center].max():.4f} (uniform would be {1 / N:.4f})')

## Section 12: Grad-CAM analysis (post-hoc explanations)

Figures compare the baseline with the selected model on the same images, then show every model on one image in the layout the sheet recommends (Input | Baseline | SE | ECA | CBAM | Self-Attention), plus the position comparison.

In [ ]:
# Baseline vs selected Grad-CAM on the representative images
n_rep = len(REP_IDX)
fig, axes = plt.subplots(n_rep, 4, figsize=(13, 3.0 * n_rep), squeeze=False)
for k in range(n_rep):
    img = X_test[VIS_IDX[k]].astype(np.uint8)
    cb = VIS[BASE]['cams'][k]; cs = VIS[SELECTED]['cams'][k]
    axes[k, 0].imshow(img); axes[k, 0].set_title(f'true: {VIS_LABEL[k]}', fontsize=8); axes[k, 0].axis('off')
    overlay(axes[k, 1], img, cb, f'{BASE}\n{panel_title(BASE, k)}')
    overlay(axes[k, 2], img, cs, f'{SELECTED}\n{panel_title(SELECTED, k)}')
    axes[k, 3].imshow(cs - cb, cmap='bwr', vmin=-1, vmax=1); axes[k, 3].set_title('selected minus baseline CAM', fontsize=8)
    axes[k, 3].axis('off')
fig.tight_layout(); savefig(fig, 'F37_gradcam_baseline_vs_selected.png')

# Recommended layout (sheet): Input | Baseline | SE | ECA | CBAM | Self-Attention (+ Coordinate, Combined)
LAYOUT = [(BASE, 'Baseline'), ('F1_SE', 'SE'), ('F2_ECA', 'ECA'), ('F3_CBAM', 'CBAM'),
          ('A4_Self', 'Self-Attention'), ('F4_Coordinate', 'Coordinate'), ('A5_Combined', 'Combined')]
k = 0                                                     # representative image
fig, axes = plt.subplots(1, 1 + len(LAYOUT), figsize=(3.0 * (1 + len(LAYOUT)), 3.4))
img = X_test[VIS_IDX[k]].astype(np.uint8)
axes[0].imshow(img); axes[0].set_title(f'Input\ntrue: {VIS_LABEL[k]}', fontsize=8); axes[0].axis('off')
for j, (run, lab) in enumerate(LAYOUT):
    overlay(axes[j + 1], img, VIS[run]['cams'][k], f'{lab}\n{panel_title(run, k)}')
fig.tight_layout(); savefig(fig, 'F38_gradcam_recommended_layout.png')

# Grad-CAM across insertion positions (same image): P1, P2, P3, P4 and multi-stage
POS_RUNS = [('none', BASE), ('P1', 'B1_P1'), ('P2', 'B2_P2'), ('P3', 'A3_ChannelSpatial'), ('P4', 'B4_P4'),
            ('P2+P3+P4', 'C3_P2P3P4')]
fig, axes = plt.subplots(2, len(POS_RUNS), figsize=(2.9 * len(POS_RUNS), 6.0), squeeze=False)
for j, (lab, run) in enumerate(POS_RUNS):
    for r, k in enumerate([0, 1]):
        img = X_test[VIS_IDX[k]].astype(np.uint8)
        if r == 0:
            overlay(axes[r, j], img, VIS[run]['cams'][k], f'{lab}: {run}\n{panel_title(run, k)}')
        else:
            overlay(axes[r, j], img, VIS[run]['cams'][k], f'{lab}\n{panel_title(run, k)}')
fig.tight_layout(); savefig(fig, 'F39_gradcam_by_position.png')

# Attention-type Grad-CAM for every unique run on two images (a visual review of all runs)
RUN_ORDER = list(RUN_SPECS)
n_r = len(RUN_ORDER); cols = 6; rows = int(np.ceil(n_r / cols))
fig, axes = plt.subplots(rows, cols, figsize=(3.0 * cols, 3.1 * rows), squeeze=False)
for ax, run in zip(axes.flat, RUN_ORDER):
    img = X_test[VIS_IDX[0]].astype(np.uint8)
    overlay(ax, img, VIS[run]['cams'][0], f'{run}\n{panel_title(run, 0)}', cmap='jet')
for ax in list(axes.flat)[n_r:]:
    ax.axis('off')
fig.tight_layout(); savefig(fig, 'F40_gradcam_all_runs.png')

## Section 13: Error analysis and qualitative comparison

Confidence distributions show whether attention changes how sure the network is. The qualitative grid compares baseline, selected, and several mechanisms on the same images, and the last row marks the images that attention fixes or breaks.

In [ ]:
# Confidence distributions and entropy: baseline vs selected (test set)
conf_b = PROBS[BASE].max(1); conf_s = PROBS[SELECTED].max(1)
ent = lambda P: -np.sum(P * np.log(np.clip(P, 1e-9, 1)), axis=1)
fig, axes = plt.subplots(1, 2, figsize=(11, 4))
axes[0].hist(conf_b, bins=30, alpha=0.55, label=BASE); axes[0].hist(conf_s, bins=30, alpha=0.55, label=SELECTED)
axes[0].set_xlabel('top-1 confidence'); axes[0].set_ylabel('test images'); axes[0].legend(fontsize=8)
axes[0].set_title('Prediction confidence distribution')
axes[1].hist(ent(PROBS[BASE]), bins=30, alpha=0.55, label=BASE); axes[1].hist(ent(PROBS[SELECTED]), bins=30, alpha=0.55, label=SELECTED)
axes[1].set_xlabel('predictive entropy (nats)'); axes[1].legend(fontsize=8); axes[1].set_title('Predictive entropy')
savefig(fig, 'F41_confidence_entropy_distribution.png')

# Qualitative comparison grid across mechanisms
QUAL = [(BASE, 'Baseline'), (SELECTED, 'Selected'), ('A1_Channel', 'Channel'), ('A2_Spatial', 'Spatial'),
        ('A4_Self', 'Self'), ('F2_ECA', 'ECA'), ('F3_CBAM', 'CBAM')]
n_q = min(8, len(VIS_IDX))
fig, axes = plt.subplots(n_q, 1 + len(QUAL), figsize=(2.6 * (1 + len(QUAL)), 2.6 * n_q), squeeze=False)
for k in range(n_q):
    img = X_test[VIS_IDX[k]].astype(np.uint8)
    axes[k, 0].imshow(img); axes[k, 0].axis('off')
    axes[k, 0].set_title(f'{VIS_LABEL[k]} [{CASE[VIS_IDX[k]]}]', fontsize=7)
    for j, (run, lab) in enumerate(QUAL):
        ok = VIS[run]['preds'][k] == y_test[VIS_IDX[k]]
        overlay(axes[k, 1 + j], img, VIS[run]['cams'][k],
                f"{lab}: {CLASS_NAMES[VIS[run]['preds'][k]]}", cmap='jet')
        axes[k, 1 + j].set_title(f"{lab}: {CLASS_NAMES[VIS[run]['preds'][k]]}", fontsize=7,
                                 color='green' if ok else 'red')
fig.tight_layout(); savefig(fig, 'F42_qualitative_comparison_grid.png')

# Summary of fixed and broken cases (text table)
case_rows = []
for i in VIS_IDX:
    k = VIS_IDX.index(i)
    case_rows.append({'test index': i, 'case': CASE[i], 'true': CLASS_NAMES[y_test[i]],
                      BASE: CLASS_NAMES[base_pred[i]], SELECTED: CLASS_NAMES[sel_pred[i]],
                      'selected correct': bool(sel_pred[i] == y_test[i]), 'baseline correct': bool(base_pred[i] == y_test[i])})
show(save_table(pd.DataFrame(case_rows), 'table15_visualization_cases'), 'Visualization images: baseline vs selected')

## Section 14: Quantitative, complexity, and qualitative analysis (summary)

The numbers below are computed from the run results. Differences smaller than the bootstrap confidence interval, or with McNemar p >= 0.05, should not be presented as real effects.

In [ ]:
def delta(run):
    a, b = RUN_METRICS[run], RUN_METRICS[BASE]
    return dict(run=run,
                d_acc=a['test_accuracy'] - b['test_accuracy'],
                d_macro_f1=a['macro_f1'] - b['macro_f1'],
                d_params=a['total_params'] - b['total_params'],
                latency_ratio=a['inference_ms_per_image'] / b['inference_ms_per_image'],
                p_mcnemar=a['mcnemar_p_vs_baseline'],
                significant=a['mcnemar_p_vs_baseline'] < 0.05)

COMPLEX = pd.DataFrame([delta(r) for r in RUN_SPECS if r != BASE])
COMPLEX['acc gain per +1M params'] = np.where(COMPLEX['d_params'] > 0,
                                              COMPLEX['d_acc'] / (COMPLEX['d_params'] / 1e6), np.nan)
show(save_table(COMPLEX.round(5), 'table16_complexity_analysis'),
     'Complexity analysis: accuracy change vs parameter and latency cost (relative to baseline)')

def lab(run):
    return f"{run} ({RUN_METRICS[run]['test_accuracy']:.3f})"

def ranked(runs):
    return sorted(runs, key=lambda r: -RUN_METRICS[r]['test_accuracy'])

sel = RUN_METRICS[SELECTED]; base = RUN_METRICS[BASE]
ci_sel = (sel['test_acc_ci_low'], sel['test_acc_ci_high'])
ci_b = (base['test_acc_ci_low'], base['test_acc_ci_high'])
print('FINDINGS (computed from this run)')
print('-' * 72)
print(f"1. Attention type @P3: baseline {base['test_accuracy']:.4f} vs selected {SELECTED} {sel['test_accuracy']:.4f} "
      f"(delta {sel['test_accuracy'] - base['test_accuracy']:+.4f}; McNemar p = {sel['mcnemar_p_vs_baseline']:.3f}, "
      f"{'significant' if sel['mcnemar_p_vs_baseline'] < 0.05 else 'not significant'} at 5 %).")
print(f"   95 % CIs: baseline [{ci_b[0]:.3f}, {ci_b[1]:.3f}], selected [{ci_sel[0]:.3f}, {ci_sel[1]:.3f}].")
print('2. Attention type ranking (test accuracy):', ' > '.join(ranked(
      ['A0_Baseline','A1_Channel','A2_Spatial','A3_ChannelSpatial','A4_Self','A5_Combined'])))
pos_runs = {'P1': 'B1_P1', 'P2': 'B2_P2', 'P3': 'A3_ChannelSpatial', 'P4': 'B4_P4'}
print('3. Position (channel+spatial), test accuracy:', {p: round(RUN_METRICS[r]['test_accuracy'], 4) for p, r in pos_runs.items()})
print('   Best position by test accuracy (analysis only):', max(pos_runs, key=lambda p: RUN_METRICS[pos_runs[p]]['test_accuracy']))
print(f"4. Multi-stage: C1 P2+P3 {RUN_METRICS['C1_P2P3']['test_accuracy']:.4f}, "
      f"C2 P3+P4 {RUN_METRICS['C2_P3P4']['test_accuracy']:.4f}, C3 P2+P3+P4 {RUN_METRICS['C3_P2P3P4']['test_accuracy']:.4f} "
      f"vs single-stage P3 {RUN_METRICS['A3_ChannelSpatial']['test_accuracy']:.4f} / P4 {RUN_METRICS['B4_P4']['test_accuracy']:.4f}. "
      f"Extra parameters of C3 over P3: {RUN_METRICS['C3_P2P3P4']['total_params'] - RUN_METRICS['A3_ChannelSpatial']['total_params']:,}.")
print(f"5. Order: channel->spatial {RUN_METRICS['A3_ChannelSpatial']['test_accuracy']:.4f} vs "
      f"spatial->channel {RUN_METRICS['D2_SpatialChannel']['test_accuracy']:.4f} "
      f"(difference {RUN_METRICS['D2_SpatialChannel']['test_accuracy'] - RUN_METRICS['A3_ChannelSpatial']['test_accuracy']:+.4f}).")
print(f"6. Kernel: 3x3 {RUN_METRICS['E1_Kernel3']['test_accuracy']:.4f}, 5x5 {RUN_METRICS['E2_Kernel5']['test_accuracy']:.4f}, "
      f"7x7 {RUN_METRICS['A2_Spatial']['test_accuracy']:.4f}; latency 3x3 {RUN_METRICS['E1_Kernel3']['inference_ms_per_image']:.2f} ms, "
      f"7x7 {RUN_METRICS['A2_Spatial']['inference_ms_per_image']:.2f} ms.")
print(f"7. Established mechanisms (test accuracy): " + ', '.join(f"{r}={RUN_METRICS[r]['test_accuracy']:.4f}"
      for r in ['F1_SE', 'F2_ECA', 'F3_CBAM', 'F4_Coordinate']))
print(f"8. Sanity check: SE minus Channel = {RUN_METRICS['F1_SE']['test_accuracy'] - RUN_METRICS['A1_Channel']['test_accuracy']:+.4f} "
      f"(same block; the spread indicates the pipeline noise floor).")
print(f"9. Self-attention: {RUN_METRICS['A4_Self']['test_accuracy']:.4f} accuracy with "
      f"{RUN_METRICS['A4_Self']['attention_params']:,} attention parameters "
      f"(channel attention at P3: {RUN_METRICS['A1_Channel']['attention_params']:,}).")
print(f"10. Most accurate run overall: {max(RUN_SPECS, key=lambda r: RUN_METRICS[r]['test_accuracy'])}; "
      f"largest accuracy gain over baseline per +1M params: "
      f"{COMPLEX.dropna(subset=['acc gain per +1M params']).sort_values('acc gain per +1M params', ascending=False)['run'].iloc[0]}.")
print(f"11. Per-class: most-improved class (recall change, selected - baseline) = "
      f"{CLASS_NAMES[int(np.argmax(PC_SEL['recall (= class accuracy)'].values - PC_BASE['recall (= class accuracy)'].values))]}; "
      f"hardest class for the selected model = {CLASS_NAMES[int(np.argmin(PC_SEL['recall (= class accuracy)'].values))]}.")
off = cm_sel - cm_base; np.fill_diagonal(off, 0)
i_, j_ = np.unravel_index(np.argmax(np.abs(off)), off.shape)
print(f"12. Largest change in confusions: true={CLASS_NAMES[i_]} predicted={CLASS_NAMES[j_]} "
      f"({off[i_, j_]:+d} images).")

### Lab questions (sheet "Questions Students Must Answer")

Each question is answered from the results in this notebook. The right-hand column names the table or figure that holds the evidence, so the written answers can be checked directly against the outputs.

In [ ]:
QUESTIONS = pd.DataFrame([
    ('Purpose of residual connections in ResNet50?', 'Architecture (Sec. 3); F03'),
    ('What does conv2_x represent vs conv5_x?', 'Table 1; F03'),
    ('Why does spatial resolution fall with depth? Why do channels grow?', 'Table 1; F03'),
    ('Low-level vs intermediate vs high-level features?', 'Table 1; feature-type column'),
    ('Purpose of an attention mechanism?', 'Section 4'),
    ('What does channel attention learn? Spatial attention?', 'Section 4; F30, F33'),
    ('How does channel attention differ from spatial attention?', 'F30 vs F33; table 1'),
    ('Why can channel and spatial attention be used sequentially?', 'Section 4; table 4; F35'),
    ('Why might channel attention behave differently at P2 vs P4?', 'table 2; F30; attention_io_dimensions'),
    ('Why can spatial attention help at an intermediate stage?', 'table 2; F33; F39'),
    ('Why is self-attention expensive at high resolution?', 'Section 4; cost table; F36'),
    ('How does the self-attention matrix size grow with resolution?', 'F36 (N x N with N = HW)'),
    ('How does accuracy vary with insertion position?', 'table 2; F06; F23'),
    ('How does attention at P2 differ visually from P4?', 'F33; F39'),
    ('Does attention at multiple stages always help?', 'table 3; F07; complexity analysis'),
    ('Does channel->spatial behave differently from spatial->channel?', 'table 4; F08; F35'),
    ('How does spatial kernel size influence the map?', 'table 5; F09; F34'),
    ('How does SE differ from ECA?', 'table 6; F10; F30'),
    ('How does CBAM combine channel and spatial attention?', 'Section 4; F10'),
    ('How does self-attention differ from channel attention?', 'mechanism table; F36'),
    ('Does the largest-parameter mechanism give the highest accuracy?', 'F14; complexity analysis'),
    ('Internal attention map vs Grad-CAM?', 'Section 11 introduction; F29-F36 vs F37-F40'),
    ('Why is Grad-CAM post-hoc?', 'Section 12 introduction'),
    ('Does the model suppress background regions?', 'F37; F38; F40'),
    ('Do attention maps differ for correct vs incorrect predictions?', 'F37; F42; table 15'),
    ('Which flower classes benefit most from attention?', 'F18; table 9 vs table 8'),
    ('What does the confusion matrix show that accuracy does not?', 'F26a/b, F27, F28'),
    ('Why is macro F1 useful in addition to accuracy?', 'F16; per-class table; F19'),
    ('What is the accuracy-complexity trade-off?', 'F14; table 16'),
    ('What is the F1-inference-time trade-off?', 'F15; table 16'),
    ('Why must every ablation use the same split?', 'Section 2; protocol table'),
    ('Why keep the same training protocol across mechanisms?', 'protocol table; Section 6'),
    ('What conclusions follow from the Grad-CAM comparison?', 'F37, F39, F40 (discuss with several images)'),
], columns=['question', 'evidence in this notebook'])
show(save_table(QUESTIONS, 'lab_questions_with_evidence'), 'Lab questions and where the evidence is')

## Section 15: Requirements checklist and artifact index

The checklist below is generated from the files on disk, so each entry is marked done only if its output was actually produced. It covers every item in the sheet's required visualizations, required tables, observations, and minimum mandatory ablation set.

In [ ]:
def fig_exists(name):
    return os.path.exists(f'results/figures/{name}')

def tab_exists(name):
    return os.path.exists(f'results/tables/{name}.csv')

REQUIRED = [
    ('Required visualization: sample images, all five classes', fig_exists('F01_sample_images_all_classes.png')),
    ('Required visualization: training and validation accuracy curves', fig_exists('F21_exp1_accuracy_curves.png') and fig_exists('F25_baseline_vs_selected_curves.png')),
    ('Required visualization: training and validation loss curves', fig_exists('F22_exp1_loss_curves.png')),
    ('Required visualization: confusion matrix (baseline and selected)', fig_exists('F26a_confusion_matrix_baseline.png') and fig_exists('F26b_confusion_matrix_selected.png')),
    ('Required visualization: feature map before and after attention', fig_exists('F29_feature_before_after_attention.png')),
    ('Required visualization: channel-attention weights', fig_exists('F30_channel_weights_topK.png')),
    ('Required visualization: spatial-attention heatmap and overlay', fig_exists('F33_spatial_attention_P2_P3_P4.png')),
    ('Required visualization: self-attention response (self-attention experiment)', fig_exists('F36_self_attention_response.png')),
    ('Required visualization: baseline Grad-CAM', fig_exists('F37_gradcam_baseline_vs_selected.png')),
    ('Required visualization: Grad-CAM for an attention model', fig_exists('F38_gradcam_recommended_layout.png')),
    ('Required visualization: attention maps at P2, P3, P4', fig_exists('F33_spatial_attention_P2_P3_P4.png')),
    ('Required visualization: graphical comparison of mechanisms', fig_exists('F12_mechanism_vs_accuracy.png')),
    ('Graphical analysis: accuracy vs parameters', fig_exists('F14_accuracy_vs_params.png')),
    ('Graphical analysis: macro F1 vs inference time', fig_exists('F15_f1_vs_inference_time.png')),
    ('Table 1: ResNet50 stages', tab_exists('table1_resnet50_stages')),
    ('Table 2: candidate positions', tab_exists('table2_candidate_positions')),
    ('Sheet Table 3: model composition', tab_exists('sheet_table3_model_composition')),
    ('Sheet Table 4: position experiment B1-B4 including P1', tab_exists('table2_exp2_position')),
    ('Sheet Table 5: spatial kernel size', tab_exists('sheet_table5_kernel_size')),
    ('Sheet Table 6: mechanism comparison', tab_exists('sheet_table6_mechanism_comparison')),
    ('Sheet Table 7: required main result table', tab_exists('main_result_table')),
    ('Per-experiment tables for Exp 1-5 and additional (all columns)', all(tab_exists(t) for t in [
        'table1_exp1_attention_type', 'table2_exp2_position', 'table3_exp3_multiposition',
        'table4_exp4_order', 'table5_exp5_kernel', 'table6_additional_mechanisms'])),
    ('Observation: attention input/output feature dimensions', tab_exists('observations_attention_io_dimensions')),
    ('Metrics: accuracy, macro P/R/F1, params, inference time (all runs)', tab_exists('master_all_models')),
    ('Confusion matrices for every run', fig_exists('F28_confusion_matrices_all_runs.png')),
    ('Minimum mandatory ablation set: baseline, Ch@P3, Sp@P3, Ch+Sp@P2/P3/P4, Self@P3 trained',
        all(r in RUN_METRICS for r in ['A0_Baseline', 'A1_Channel', 'A2_Spatial', 'A3_ChannelSpatial',
                                       'B2_P2', 'B4_P4', 'A4_Self'])),
    ('Minimum mandatory set: Grad-CAM baseline vs model and spatial maps P2/P3/P4', fig_exists('F37_gradcam_baseline_vs_selected.png') and fig_exists('F33_spatial_attention_P2_P3_P4.png')),
    ('Minimum mandatory set: graphical comparison of attention models', fig_exists('F12_mechanism_vs_accuracy.png')),
]
check = pd.DataFrame(REQUIRED, columns=['requirement', 'produced'])
check['produced'] = check['produced'].map(lambda v: 'yes' if v else 'NO')
show(save_table(check, 'checklist_status'), 'Requirements checklist (generated from disk)')
missing = check[check['produced'] == 'NO']
print('All required items produced.' if missing.empty else f'Missing: {missing["requirement"].tolist()}')

figs = sorted(os.listdir('results/figures')); tabs = sorted(os.listdir('results/tables'))
print(f'\nArtifacts: {len(figs)} figures in results/figures/, {len(tabs)} tables in results/tables/')
print('Trained models (weights):', sorted(os.listdir('results/models')))

---
### Summary of changes from the six original notebooks

- **Part 3 builders were missing.** `build_model` (with the clone-based insertion) now exists, and it is used everywhere.
- **Augmentation is now applied** in training, identically for every model. In the original it was defined and then never used.
- **Models are saved as weights** and rebuilt from code. This removes the custom-layer loading failure in Parts 4 and 6.
- **Part 6 `NameError` fixed.** Models are reloaded explicitly, not reused after deletion.
- **Class names are read from the dataset metadata**, not hard-coded.
- **Model selection uses validation accuracy**, not test accuracy, and the baseline is excluded from the "selected attention model".
- **Duplicate configurations are trained once** and reported in every table where they apply.
- **B1 (P1) is included**, as in the sheet's Table 4.
- **Two-phase fine-tuning** follows the sheet (frozen backbone, then fine-tune conv5_x at a lower learning rate).
- **Statistics added:** bootstrap 95 % CIs and exact McNemar tests against the baseline, so small differences are not over-interpreted.
- **Resumable training** and a single-process evaluation that keeps memory bounded.